# Zzzz… é Aedes? — detector ternário de mosquitos pelo som

**Luiz Eduardo do Nascimento Ciarallo · IFSC · IA na Eletrônica · 2026**

Este caderno é a **única fonte da verdade** do projeto e da apresentação. Tudo o que aparece nos slides sai daqui: dos arquivos de áudio até a matriz de confusão, sem importar nenhum script do repositório.

**A pergunta.** Dada uma gravação curta, o som é *ruído*, *outro mosquito* ou *Aedes aegypti*? (Detector ternário.)

**O caminho, na ordem dos slides**

| § | O que acontece | Por quê |
|---|---|---|
| 1 | Configuração | tudo reproduzível, com sementes fixas |
| 2 | Dados: quem ensina o quê | a escolha dos exemplos é a primeira decisão do projeto |
| 3 | A física do zumbido | entender o sinal antes de modelar |
| 4 | Do áudio a 80 números | o modelo não ouve; ele lê números |
| 5 | Montar a tabela de atributos | uma linha por gravação, com fonte e grupo |
| 6–8 | Preparo e prova honesta | normalizar por fonte, pesos, validação por grupo |
| 9–10 | Os modelos e o ranking de 9 | escolher o modelo com a mesma prova |
| 11–13 | Modelo final: erros, ROC e limiar | medir de três formas e escolher o corte |
| 14 | A prova mais justa | Aedes × outros mosquitos no mesmo aparelho |
| 15–16 | O que o modelo ouve | interpretabilidade e teste de origem |
| 17 | Aedes de fontes nunca vistas | até onde o resultado vale |
| 18 | Aplicar a um WAV | usar o modelo na prática |

**Como rodar.** *Run All*. Na primeira vez o caderno lê os áudios em `data/` (alguns minutos), grava caches em `cache_notebook/` e treina os modelos (dezenas de minutos, mais se não houver GPU). Nas vezes seguintes usa os caches. Para refazer uma etapa, apague o arquivo correspondente em `cache_notebook/` ou mude os controles da célula 1.

**O que o caderno lê de fora (dados, não código):** os áudios em `data/raw` e `data/external`, o manifesto `data/metadata/ternary_segments.csv.gz`, o mapa de duplicatas `data/metadata/known_recording_groups.csv` e os tensores publicados por Torres et al. (`data/external/torres_2025`).

> **Honestidade sobre o alcance.** Os resultados são **retrospectivos**: usam dados já conhecidos do projeto, não são um teste cego. A detecção de Aedes reflete bem o reconhecimento das duas fontes de Aedes usadas no treino (Torres e Funchal); Aedes gravados em outros aparelhos ainda não são reconhecidos (§17).

**Termos desta seção**

- **Aprendizado de máquina (ML)**: programas que aprendem padrões a partir de exemplos, em vez de seguir regras escritas à mão.
- **Detector ternário**: classificador com três respostas possíveis: ruído, outro mosquito ou Aedes.
- **Classe**: cada resposta possível do detector (aqui: ruído, outro mosquito, Aedes).
- **Aedes aegypti**: o mosquito que transmite dengue, zika e chikungunya; é o alvo do detector.
- **Dataset (base de dados)**: coleção de gravações com a resposta correta conhecida de cada uma.
- **Rótulo**: a resposta correta conhecida de um exemplo (aqui, a classe dele).
- **Modelo**: o programa que, depois de ajustado com exemplos, dá a resposta para gravações novas.
- **Treino e prova (teste)**: treino é o ajuste do modelo com exemplos; prova é a avaliação em exemplos que ele não viu.
- **Fonte da verdade**: o lugar que vale quando há divergência: os slides seguem este caderno, e não o contrário.
- **Retrospectivo × teste cego**: retrospectivo = avaliado em dados que o projeto já conhecia; teste cego = avaliado em dados novos, escolhidos depois de congelar o método.
- **Cache**: cópia guardada de um resultado caro de calcular, para não recalcular toda vez.

## 1. Configuração

**O que faz:** importa as bibliotecas, localiza a raiz do projeto e fixa as sementes.
**Também mostra a configuração da GPU** (modelo, memória, versão do PyTorch e do ROCm), porque o treino das redes neurais depende dela e quem for reproduzir precisa saber em que máquina os números saíram.
**Por quê:** resultados de ML precisam ser reproduzíveis. As sementes `11, 22, 33` definem os sorteios das 3 repetições da validação cruzada; os caminhos são calculados a partir da pasta do projeto, então o caderno funciona em qualquer cópia.

**Termos desta seção**

- **Semente (seed)**: número que fixa um sorteio aleatório: com a mesma semente, o sorteio sai igual e o resultado se repete.
- **Reprodutível**: quem rodar de novo, com os mesmos dados e sementes, obtém o mesmo resultado.
- **CPU e GPU**: CPU é o processador comum; GPU é a placa gráfica, que acelera o treino de redes neurais.
- **ROCm / HIP / CUDA**: plataformas que deixam o PyTorch usar a GPU: CUDA é a da NVIDIA; ROCm (e a interface HIP) é a da AMD.
- **Memória da GPU (VRAM)**: memória da própria placa gráfica; o treino precisa que os dados e a rede caibam nela.
- **ffmpeg**: programa que decodifica formatos de áudio que a biblioteca principal não abre (como .m4a).
- **Biblioteca**: conjunto de funções prontas (NumPy para números, scikit-learn para modelos, PyTorch para redes neurais).

In [1]:
import collections, hashlib, json, os, re, subprocess, time, warnings
from math import gcd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, matplotlib.pyplot as plt, soundfile as sf
import torch, torch.nn as nn
from scipy.signal import resample_poly
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             balanced_accuracy_score)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

# --- raiz do projeto: a pasta que contém data/metadata/ternary_segments.csv.gz ---
RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'data/metadata/ternary_segments.csv.gz').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Abra este caderno dentro de uma cópia do projeto mosquito-wingbeat.')
MANIFESTO = RAIZ / 'data/metadata/ternary_segments.csv.gz'        # lista de janelas de 1 s de cada áudio
MAPA_DUPLICATAS = RAIZ / 'data/metadata/known_recording_groups.csv'  # gravações já sabidamente repetidas
TORRES_JSON = RAIZ / 'data/external/torres_2025/Dataset/Melspec.json'  # espectrogramas publicados por Torres et al.
CACHE = RAIZ / 'cache_notebook'; CACHE.mkdir(exist_ok=True)

# --- controles ---
RECALCULAR_ATRIBUTOS = False   # True: relê todos os áudios e refaz a tabela de atributos
RECALCULAR_RANKING = False     # True: retreina os 9 modelos (demora)
SEMENTES = [11, 22, 33]        # 3 repetições da validação cruzada
CLASSES = ['ruído', 'outro mosquito', 'Aedes']
FONTES_DE_AEDES = ['torres', 'funchal']   # só estas duas fontes ensinam o que é Aedes (ver §2)

torch.set_num_threads(min(8, os.cpu_count() or 1))
DISPOSITIVO = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.enabled = False   # evita falha conhecida da LSTM em algumas GPUs ROCm; a versão nativa do PyTorch funciona
print('Raiz:', RAIZ)
print('PyTorch', torch.__version__, '| ROCm/HIP', torch.version.hip, '| CUDA', torch.version.cuda, '| threads de CPU:', torch.get_num_threads())
if DISPOSITIVO == 'cuda':
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU em uso: {gpu.name} | memória {gpu.total_memory / 1e9:.1f} GB | unidades de processamento: {gpu.multi_processor_count} | '
          f'arquitetura: {getattr(gpu, "gcnArchName", "n/d")} | placas visíveis: {torch.cuda.device_count()}')
    print('As redes neurais (MLP, CNN, LSTM, Bi-LSTM, DANN) treinam e prevêem nesta GPU; SVM, árvores e regressão logística rodam na CPU, porque o scikit-learn não usa GPU.')
else:
    print('ATENÇÃO: nenhuma GPU visível. As redes neurais vão treinar na CPU, bem mais devagar. Use um ambiente do PyTorch com suporte à sua GPU.')

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/lciarallo/Documentos/Git_projects/mosquito-wingbeat/.venv/lib64/python3.15/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
  File "/tmp/ipykernel_112145/3114574413.py", line 6, in <module>
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, soundfile as sf
ModuleNotFoundError: No module named 'pandas'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/lciarallo/Documentos/Git_projects/mosquito-wingbeat/.venv/lib64/python3.15/site-packages/IPython/core/interactiveshell.py", line 2279, in showtraceback
  File "/home/lciarallo/Documentos/Git_projects/mosquito-wingbeat/.venv/lib64/python3.15/site-packages/IPython/core/ultratb.py", line 1300, in structured_traceback
  File "/home/lciarallo/Documentos/Git_projects/mosquito-wingbeat/.venv/lib64/python3.15/site-packages/IPython/core/ultratb.py", line 1166, in structured_traceback
  File "/home

### Ambiente em que os números deste caderno foram obtidos

Para reproduzir, use um ambiente equivalente. **Testado com:** Python 3.11, NumPy 2.4, pandas 3.0, SciPy 1.17, scikit-learn 1.9, Matplotlib 3.11, SoundFile 0.14 (libsndfile 1.2), PyTorch 2.9.1 com ROCm 6.4 (GPU AMD) e ffmpeg 9.

```
pip install numpy pandas scipy scikit-learn matplotlib soundfile
pip install torch            # escolha em pytorch.org a versão da sua GPU (CUDA para NVIDIA, ROCm para AMD)
# ffmpeg: instale pelo gerenciador de pacotes do sistema (ex.: sudo dnf install ffmpeg)
```

A célula abaixo registra o que **está** instalado nesta execução. Redes neurais em GPU não são bit a bit idênticas entre máquinas: espere diferenças pequenas nas redes (as sementes fixam os sorteios, não a aritmética da placa).

In [ ]:
import platform, shutil, sklearn, scipy, matplotlib
print(f'Python {platform.python_version()} | NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__} | '
      f'scikit-learn {sklearn.__version__} | Matplotlib {matplotlib.__version__} | SoundFile {sf.__version__} (libsndfile {sf.__libsndfile_version__})')
print('ffmpeg:', shutil.which('ffmpeg') or 'NÃO ENCONTRADO: os arquivos .m4a do Dryad não poderão ser lidos')

## 2. Dados: quem ensina o quê

Um modelo de ML aprende por **exemplos rotulados**. A escolha dos exemplos é a primeira decisão de projeto:

1. **Aedes** vêm de dois datasets grandes: *Torres* e *Funchal*.
2. Os outros datasets ensinam o que **não** é Aedes: outros mosquitos e ruídos.
3. Cada exemplo guarda **de onde veio** (fonte e aparelho). Isso serve depois para normalizar e para montar a prova.

O manifesto lista janelas de áudio (arquivo, instante inicial, volume `rms`, fonte). Primeiro damos a cada janela a classe do projeto: `0` ruído, `1` outro mosquito, `2` Aedes.

**Termos desta seção**

- **Fonte (coleção)**: o conjunto de gravações de onde um exemplo veio. Aqui: Dryad (repositório público com gravações de várias espécies), HumBug (gravações de mosquitos feitas com celulares), ESC-50 (50 tipos de sons ambientais, usados como ruído), Funchal (Aedes e outros mosquitos gravados em vários aparelhos, Vasconcelos et al., 2020), MosquitoSong (gravações do estudo MosquitoSong+), UFRGS (Aedes gravados com aplicativos de celular, Paim et al., 2023) e Torres (espectrogramas de Aedes publicados por Torres et al., 2025).
- **Manifesto**: tabela que lista cada janela de áudio: arquivo, instante inicial, volume e fonte.
- **Janela**: trecho curto do áudio (cerca de 1 s) analisado de uma vez.
- **Volume (rms)**: raiz da média dos quadrados das amostras: mede o quão "alto" é um trecho.
- **Taxa de amostragem (Hz, kHz)**: quantas medidas do som são guardadas por segundo. Em 8 kHz são 8.000 por segundo, e só cabem frequências até a metade disso, 4 kHz.
- **Amostra**: um valor numérico do sinal de áudio em um instante.
- **Reamostrar**: trocar a taxa de amostragem de um áudio.
- **Mono**: áudio de um único canal (os estéreo são somados).
- **PCM**: a forma digital do áudio: a sequência de amostras numéricas.
- **Impressão digital (hash, SHA-256)**: código curto calculado a partir do conteúdo: conteúdos iguais dão o mesmo código.
- **Duplicata e bloco**: gravações com o mesmo som em arquivos diferentes. Um bloco junta todos os grupos que repetem o mesmo áudio.
- **União-busca (union-find)**: algoritmo que junta itens ligados entre si em conjuntos.
- **Grupo**: conjunto de exemplos que não podem ficar dos dois lados da prova (ver §7).

In [ ]:
manifesto = pd.read_csv(MANIFESTO, low_memory=False)
rotulo = manifesto.label.astype(str).str.lower()
# ruído = rótulo "noise" ou qualquer som do ESC-50 (sons ambientais); Aedes = rótulo Aedes aegypti; o resto é "outro mosquito"
manifesto['classe'] = np.where(rotulo.eq('noise') | manifesto.domain.eq('esc50'), 0,
                               np.where(rotulo.eq('aedes aegypti'), 2, 1))
manifesto['qualidade_ok'] = manifesto.quality_ok.isin([True, 'True', 'true', 1, '1'])
manifesto['desconhecido'] = rotulo.isin(['unknown_mosquito', 'unknown', 'mosquito'])

resumo = (manifesto.groupby(['domain', 'classe']).agg(janelas=('path', 'size'), arquivos=('path', 'nunique'))
          .rename(index={0: 'ruído', 1: 'outro mosquito', 2: 'Aedes'}, level='classe'))
resumo

### De onde vêm os áudios e como conferir que estão no lugar

O caderno **não baixa** os dados; ele lê pastas que já existem no projeto. A tabela diz onde obter cada coleção e a célula seguinte confere, arquivo por arquivo, que tudo o que o manifesto cita está presente. Se faltar algo, ela para com a lista do que falta.

| Coleção | Onde obter | Pasta no projeto |
|---|---|---|
| Dryad / Abuzz (Mukundarajan et al., 2017) | doi 10.5061/dryad.98d7s (cópia no Zenodo, registro 4964774) | `data/raw` |
| Ruído de apoio (Altayeb et al.) | github.com/Mjrovai/wingbeat-mosquito-tinyml (`dataset/noise.zip`) | `data/noise` |
| HumBugDB (Kiskin et al., 2021) | doi 10.5281/zenodo.4904800 · github.com/HumBug-Mosquito/HumBugDB | `data/external/humbug`, `data/external/reliability_2026_10_01`, `data/external/reproduction_*` |
| ESC-50 (Piczak, 2015) | github.com/karolpiczak/ESC-50 | `data/external/reliability_2026_10_01` |
| Funchal (Vasconcelos et al., 2020) | figshare ligado ao artigo *Scientific Data* 7:382 | `data/external/additional/figshare` |
| MosquitoSong+ (Supratak et al., 2024) | github.com/akaraspt/mosquitosongp | `data/external/additional/mosquitosong` |
| UFRGS (Paim et al., 2023) | github.com/ComputerNetworks-UFRGS/Aedes_Detector_Smartphone_App_Paper | `data/external/additional/ufrgs` |
| Torres (Torres et al., 2025) | github.com/NakanoMariko/Dengue-Infected-Mosquito-Detection | `data/external/torres_2025` |

Os caminhos escritos no manifesto são relativos à raiz do projeto, então a **estrutura de pastas precisa ser a mesma**.

In [ ]:
def verificar_dados():
    """Falha cedo, com a lista do que falta, se algum arquivo citado pelo manifesto (ou os dois auxiliares) não existir."""
    caminhos = manifesto.path.drop_duplicates().tolist()
    faltando = [p for p in caminhos if not (RAIZ / p).is_file()]
    faltando += [str(p.relative_to(RAIZ)) for p in (TORRES_JSON, MAPA_DUPLICATAS) if not p.is_file()]
    if faltando:
        raise FileNotFoundError(f'{len(faltando)} arquivos faltando (veja a tabela acima para obter os dados). Primeiros: {faltando[:5]}')
    print(f'{len(caminhos):,} arquivos de áudio do manifesto + Torres + mapa de duplicatas: todos presentes')
verificar_dados()

### Regras de montagem do corpus

Quatro regras, todas escritas **antes** de olhar qualquer resultado:

* **Corpus original** (Dryad, HumBug, ESC-50): só janelas com qualidade verificada e rótulo conhecido. Vira **uma linha por grupo de gravação**, resumindo as **3 janelas mais energéticas** (as que realmente têm o mosquito ou o ruído, não o silêncio entre elas).
* **Fontes externas** (Funchal, MosquitoSong, UFRGS): **uma linha por arquivo**, média de todas as janelas.
* **Torres**: 300 espectrogramas publicados (todos Aedes, infectados ou não; o modelo **não** identifica infecção). Grupos de 10 espectrogramas consecutivos, porque não há identificação de indivíduos.
* **Aedes só de Torres e Funchal.** Aedes das outras fontes ficam de fora do treino e da prova principal, e são usados no final (§17) como **teste de fontes nunca vistas**.

**Grupo** = o que não pode ficar dos dois lados da prova. No corpus original, gravações idênticas em arquivos diferentes (duplicatas) precisam ficar no mesmo grupo; senão o modelo "reconhece" na prova algo que já viu no treino. Detectamos duplicatas comparando o conteúdo PCM dos áudios.

In [ ]:
N_JANELA = {'long': 7936, 'short': 2400}   # amostras a 8 kHz: 0,992 s (janela longa) e 0,3 s (arquivo curto)

def ler_audio_8k(caminho):
    """Lê o áudio como mono a 8 kHz. Formatos que o soundfile não abre (ex.: .m4a) passam pelo ffmpeg."""
    try:
        x, taxa = sf.read(str(caminho), dtype='float32', always_2d=True); x = x.mean(1)
    except sf.LibsndfileError:
        bruto = subprocess.run(['ffmpeg', '-v', 'error', '-i', str(caminho), '-ac', '1', '-ar', '8000', '-f', 'f32le', 'pipe:1'],
                               check=True, capture_output=True).stdout
        return np.frombuffer(bruto, dtype='<f4').copy()
    if taxa != 8000:
        d = gcd(8000, taxa); x = resample_poly(x, 8000 // d, taxa // d).astype('float32')
    if not len(x) or not np.isfinite(x).all():
        raise ValueError(f'áudio vazio ou inválido: {caminho}')
    return x

def assinaturas_pcm(x):
    """Impressões digitais do conteúdo: o áudio inteiro e pedaços alinhados de 1 s, 0,992 s e 0,3 s,
    como estão e com volume normalizado. Dois arquivos com a mesma impressão têm o mesmo som."""
    chaves, pedacos = set(), [x]
    for seg in (1.0, 0.992, 0.3):
        n = int(round(seg * 8000)); pedacos.extend(x[i:i + n] for i in range(0, len(x), n))
    for p in pedacos:
        for v in (p, p / max(float(np.abs(p).max()), 1e-12) * 0.9):
            pcm = np.rint(np.clip(v, -1, 32767 / 32768) * 32768).astype('<i2')
            if np.ptp(pcm.astype('int32')) > 2:          # ignora trechos quase constantes (silêncio digital)
                chaves.add(f'{len(pcm)}:' + hashlib.sha256(pcm.tobytes()).hexdigest())
    return chaves

def blocos_de_duplicatas(quadro):
    """Une (união-busca) os grupos que compartilham alguma impressão PCM ou que o mapa de duplicatas já liga."""
    arquivos = quadro.groupby('path').group.first()
    with ThreadPoolExecutor(8) as ex:
        sigs = list(ex.map(lambda p: (p, assinaturas_pcm(ler_audio_8k(RAIZ / p))), arquivos.index))
    grupos = sorted(quadro.group.astype(str).unique()); pai = {g: g for g in grupos}
    def achar(g):
        while pai[g] != g: pai[g] = pai[pai[g]]; g = pai[g]
        return g
    def unir(a, b):
        a, b = achar(a), achar(b)
        if a != b: pai[max(a, b)] = min(a, b)
    mapa = pd.read_csv(MAPA_DUPLICATAS); mapa = dict(zip(mapa.group.astype(str), mapa.collision_group.astype(str)))
    anterior = {}
    for g in grupos:
        bloco = mapa.get(g, g)
        if bloco in anterior: unir(g, anterior[bloco])
        else: anterior[bloco] = g
    dono = {}
    for p, chaves in sigs:
        g = str(arquivos[p])
        for k in chaves:
            if k in dono: unir(g, dono[k])
            else: dono[k] = g
    return {g: 'pcm:' + achar(g) for g in grupos}

# --- corpus original: só dryad/humbug/esc50, qualidade ok e rótulo conhecido ---
original = manifesto[manifesto.domain.isin(['dryad', 'humbug', 'esc50']) & manifesto.qualidade_ok & ~manifesto.desconhecido].copy()
arq_blocos = CACHE / 'blocos_pcm.json'
if arq_blocos.exists():
    blocos = json.loads(arq_blocos.read_text())
else:
    t0 = time.time(); blocos = blocos_de_duplicatas(original); arq_blocos.write_text(json.dumps(blocos))
    print(f'duplicatas verificadas em {time.time() - t0:.0f} s')
original['bloco'] = original.group.astype(str).map(blocos)
print(f'{original.group.nunique()} grupos de gravação -> {original.bloco.nunique()} blocos sem repetição '
      f'({original.group.nunique() - original.bloco.nunique()} grupos eram cópias de outro)')

Agora a escolha das **3 janelas mais energéticas de cada grupo** e a classe do grupo (a maior classe entre as suas janelas: um grupo com qualquer janela Aedes é Aedes).

In [ ]:
selecionadas = (original.sort_values(['group', 'rms'], ascending=[True, False], kind='stable')
                .groupby('group', sort=False).head(3).sort_values('group', kind='stable').reset_index(drop=True))
grupos_orig = (selecionadas.groupby('group', sort=True).agg(dominio=('domain', 'first'), bloco=('bloco', 'first')).reset_index())
grupos_orig['classe'] = grupos_orig.group.map(original.groupby('group').classe.max()).to_numpy()
print(len(selecionadas), 'janelas escolhidas em', len(grupos_orig), 'grupos')
grupos_orig.groupby(['dominio', 'classe']).size().unstack(fill_value=0).rename(columns={0: 'ruído', 1: 'outro', 2: 'Aedes'})

## 3. A física do zumbido

Antes de modelar, olhe o sinal. A asa do mosquito bate centenas de vezes por segundo; o zumbido é um **tom fundamental** `f₀` (as batidas por segundo) e seus **harmônicos** `2·f₀`, `3·f₀`… (os "ecos" do tom).

**Como medimos `f₀`:** testamos candidatos de 250 a 1000 Hz e, para cada um, olhamos a energia no candidato e nos dois primeiros harmônicos. Usamos a **média dos logaritmos** (média geométrica): um candidato só pontua alto se **os três** picos existem, o que evita confundir um pico isolado com o tom.

**Termos desta seção**

- **Frequência (Hz)**: quantas vezes por segundo algo se repete: 500 Hz = 500 ciclos por segundo.
- **Tom fundamental (f₀) e harmônicos**: f₀ é a frequência principal do zumbido (as batidas de asa por segundo); os harmônicos são múltiplos dela (2·f₀, 3·f₀…).
- **Espectro**: gráfico da energia do sinal em cada frequência.
- **FFT (transformada de Fourier)**: conta que converte um trecho de som no seu espectro.
- **Janela de Hann**: suavização aplicada às bordas do trecho antes da FFT, para evitar artefatos.
- **Média geométrica (média dos logaritmos)**: média que só é alta se todos os valores forem altos.
- **Percentil e mediana**: o percentil 10 é o valor abaixo do qual estão 10% dos dados; a mediana é o percentil 50 (o valor do meio).

In [ ]:
def janela_de(caminho, inicio_s, bucket='long'):
    onda = ler_audio_8k(RAIZ / caminho); a = int(round(inicio_s * 8000))
    return onda[a:a + N_JANELA[bucket]]

def estimar_f0(janela, taxa=8000, n_fft=16384):
    """Tom fundamental por soma harmônica (média dos logs de f0, 2·f0 e 3·f0)."""
    x = janela - janela.mean()
    espectro = np.abs(np.fft.rfft(x * np.hanning(len(x)), n=n_fft)); freqs = np.fft.rfftfreq(n_fft, 1 / taxa)
    amp = lambda f: np.interp(f, freqs, espectro)
    cand = np.arange(250, 1001)
    harm = np.exp(sum(np.log(amp(k * cand) + 1e-12) for k in range(1, 4)) / 3)
    return int(cand[np.argmax(harm)]), freqs, espectro

# uma janela de Aedes do corpus original (a de maior volume do primeiro grupo Aedes)
ex = selecionadas[selecionadas.classe.eq(2)].iloc[0]
f0, freqs, esp = estimar_f0(janela_de(ex.path, ex.start_s, ex.bucket))
plt.figure(figsize=(9, 3.4)); plt.semilogy(freqs, esp, color='tab:gray', lw=.8)
for k in range(1, 5):
    plt.axvline(k * f0, color='tab:red', ls='--', alpha=.7); plt.text(k * f0 + 8, esp.max() * .6, f'{k}·f₀' if k > 1 else 'f₀', color='tab:red')
plt.xlim(0, 2600); plt.xlabel('frequência (Hz)'); plt.ylabel('energia'); plt.title(f'Um Aedes aegypti: f₀ ≈ {f0} Hz'); plt.show()

**Só o tom basta?** Calculamos `f₀` para os grupos Aedes e para uma amostra de outros mosquitos do corpus original. Se as faixas se sobrepõem, só o tom não separa as espécies, e o modelo precisa do **espectro inteiro**, não de um número.

In [ ]:
rng = np.random.default_rng(0)
def f0_de_grupos(classe, n):
    g = selecionadas[selecionadas.classe.eq(classe)].drop_duplicates('group')
    g = g.iloc[rng.permutation(len(g))[:n]]
    return np.array([estimar_f0(janela_de(r.path, r.start_s, r.bucket))[0] for r in g.itertuples()])
f0_aedes, f0_outros = f0_de_grupos(2, 60), f0_de_grupos(1, 200)
for nome, v in [('Aedes', f0_aedes), ('outros mosquitos', f0_outros)]:
    print(f'{nome:17s} n={len(v):3d}  mediana {np.median(v):.0f} Hz  | 10–90%: {np.percentile(v, 10):.0f}–{np.percentile(v, 90):.0f} Hz')
plt.figure(figsize=(8, 3.2)); b = np.arange(250, 1051, 50)
plt.hist(f0_outros, b, alpha=.6, density=True, label='outros mosquitos'); plt.hist(f0_aedes, b, alpha=.6, density=True, color='tab:red', label='Aedes')
plt.xlabel('f₀ estimado (Hz)'); plt.legend(); plt.title('Os tons se sobrepõem'); plt.show()

## 4. Do áudio a 80 números

O modelo não ouve; ele lê números. O caminho é sempre o mesmo, para treino, prova e uso:

1. **Janela** de áudio a 8 kHz (0,992 s; arquivos curtos usam 0,3 s).
2. **Espectrograma mel**: energia por faixa de frequência ao longo do tempo. A escala *mel* tem faixas estreitas nos graves e largas nos agudos, como o ouvido.
3. **16 faixas até 4 kHz**, onde mora o zumbido (e acima do que o áudio de 8 kHz guarda).
4. **5 resumos por faixa** (média, desvio, percentis 10, 50 e 90 no tempo): 16 × 5 = **80 números**.

**Por que o espectrograma é calculado "como se" fosse 48 kHz.** Torres et al. publicaram os espectrogramas deles nesse formato (48 kHz, FFT de 1024, avanço de 256, 32 faixas mel). Para que o **nosso** áudio e os tensores deles passem pelas mesmas contas, reamostramos nossas janelas de 8 kHz para 48 kHz e usamos o mesmo banco de filtros; só as faixas abaixo de 4 kHz são mantidas, porque acima disso o áudio de 8 kHz não tem informação.

**Termos desta seção**

- **Espectrograma**: imagem da energia por frequência (vertical) ao longo do tempo (horizontal).
- **Escala mel**: escala de frequências que imita o ouvido: faixas estreitas nos graves e largas nos agudos.
- **Banco de filtros e faixa**: conjunto de filtros triangulares; cada um soma a energia de uma faixa de frequências.
- **STFT**: FFT repetida em janelinhas que deslizam pelo áudio; o resultado é o espectrograma.
- **Quadro**: cada instante (coluna) do espectrograma.
- **FFT de 1024 e avanço de 256**: a janelinha tem 1024 amostras e anda 256 amostras a cada quadro.
- **Decibel (dB)**: escala logarítmica de energia: 10·log10(potência). Valores relativos ao pico são sempre ≤ 0 dB.
- **Atributo (feature)**: cada número que descreve uma gravação para o modelo.
- **Desvio padrão**: o quanto os valores variam em torno da média.
- **Tensor**: tabela numérica com várias dimensões (aqui: gravação × quadro × faixa).

In [ ]:
FS_FRONTEND, N_FFT, SALTO, N_MEL = 48000, 1024, 256, 32

# 34 bordas em escala mel (HTK) de 0 a 24 kHz definem 32 filtros triangulares
ARESTAS = 700 * (10 ** (np.linspace(0, 2595 * np.log10(1 + FS_FRONTEND / 2 / 700), N_MEL + 2) / 2595) - 1)
_hz = np.fft.rfftfreq(N_FFT, 1 / FS_FRONTEND)
BANCO = np.zeros((N_MEL, len(_hz)), 'float32')
for i in range(N_MEL):
    BANCO[i] = np.maximum(0, np.minimum((_hz - ARESTAS[i]) / (ARESTAS[i + 1] - ARESTAS[i]),
                                         (ARESTAS[i + 2] - _hz) / (ARESTAS[i + 2] - ARESTAS[i + 1])))
    BANCO[i] /= max(BANCO[i].sum(), 1e-12)                     # cada filtro soma 1
BANCO_T, JANELA_HANN = torch.tensor(BANCO), torch.hann_window(N_FFT)
MANTER = np.flatnonzero(ARESTAS[2:] <= 4000)                   # faixas que terminam até 4 kHz
CENTROS = ARESTAS[1:17]                                        # frequência central das 16 faixas mantidas
assert len(MANTER) == 16

def espectrograma_mel_db(ondas):
    """ondas (n, amostras a 48 kHz) -> dB relativo ao pico, shape (n, quadros, 32)."""
    x = torch.as_tensor(np.asarray(ondas, 'float32'))
    x = x - x.mean(-1, keepdim=True)
    x = x / x.square().mean(-1, keepdim=True).sqrt().clamp_min(1e-6)       # volume não importa: só o formato do espectro
    stft = torch.stft(x, N_FFT, SALTO, window=JANELA_HANN, center=False, return_complex=True)
    db = 10 * torch.log10((BANCO_T @ stft.abs().square()).clamp_min(1e-12))
    db = (db - db.amax((-2, -1), keepdim=True)).clamp(-80, 0)               # relativo ao pico, piso em -80 dB
    return db.transpose(-1, -2).contiguous().numpy()

def faixas_recortadas(db):
    z = db[:, 2:-2, MANTER]                                  # descarta 2 quadros de cada borda (efeito de borda)
    return np.clip(z - z.max((1, 2), keepdims=True), -80, 0)

def descrever(db):
    """(n, quadros, 32) -> (n, 80): média, desvio e percentis 10/50/90 de cada uma das 16 faixas."""
    z = faixas_recortadas(db)
    return np.concatenate([z.mean(1), z.std(1), *np.quantile(z, [.1, .5, .9], axis=1)], axis=1).astype('float32')

T_SEQ = 32
def sequencia(db):
    """Versão "filme" dos mesmos dados para LSTM/Bi-LSTM: 32 instantes × 16 faixas."""
    z = faixas_recortadas(db); b = np.linspace(0, z.shape[1], T_SEQ + 1).round().astype(int)
    return np.stack([z[:, a:max(c, a + 1)].mean(1) for a, c in zip(b[:-1], b[1:])], 1).astype('float32')

def atributos_da_janela(janela8k):
    up = resample_poly(np.asarray(janela8k, 'float32')[None], 6, 1, axis=1)          # 8 kHz -> 48 kHz
    return espectrograma_mel_db(up)

# exemplo: o mesmo Aedes da §3
db = atributos_da_janela(janela_de(ex.path, ex.start_s, ex.bucket))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [2, 1.3]})
ax[0].imshow(faixas_recortadas(db)[0].T, origin='lower', aspect='auto', cmap='magma', extent=[0, 0.99, 0, 16])
ax[0].set_xlabel('tempo (s)'); ax[0].set_ylabel('faixa mel (0 = grave, 15 ≈ 3,2 kHz)'); ax[0].set_title('Espectrograma mel (16 faixas até 4 kHz)')
ax[1].imshow(descrever(db).reshape(5, 16), aspect='auto', cmap='viridis'); ax[1].set_yticks(range(5), ['média', 'desvio', 'p10', 'p50', 'p90'])
ax[1].set_xlabel('faixa'); ax[1].set_title('5 resumos × 16 faixas = 80 números'); plt.tight_layout(); plt.show()
print('frequências centrais das 16 faixas (Hz):', CENTROS.round().astype(int).tolist())

## 5. Montar a tabela de atributos

**O que faz:** transforma cada fonte em linhas de `X` (80 números), com classe `y`, **fonte** `fonte` e **grupo** `grupo`.

**Por quê cada decisão:**

* *Fonte* = de qual coleção veio (Dryad, HumBug, ESC-50, Funchal, MosquitoSong, UFRGS, Torres). Cada fonte tem seu aparelho e seu "sotaque", o que justifica normalizar por fonte (§7).
* *Grupo* = o que não pode estar dos dois lados da prova (§8): no corpus original, o bloco de gravações idênticas; em Torres, 10 espectrogramas consecutivos; nos outros, o **aparelho**.
* No **Funchal** o aparelho é identificado pela **taxa de amostragem da pasta** (8, 44,1 ou 48 kHz), três aparelhos distintos. Aedes, Culex e Culiseta foram gravados com os mesmos aparelhos, e é isso que dá a "prova mais justa" da §14. (Uma versão anterior deste trabalho procurava o texto "8khz" no nome da pasta; como "48khz" também o contém, os aparelhos de 8 e 48 kHz caíam no mesmo grupo. Isso foi corrigido aqui e muda os números: veja a nota ao fim da §11.)
* Torres entra pelos tensores publicados, sem refazer o espectrograma: são o único Aedes de uma fonte independente do Funchal.

**Termos desta seção**

- **Tabela de atributos**: uma linha por gravação e 80 colunas de atributos, mais classe, fonte e grupo.
- **Aparelho**: o equipamento que gravou (celular, gravador, microfone). Cada um tem o seu "sotaque".
- **Média por arquivo**: para arquivos com várias janelas, o exemplo é a média dos atributos de todas elas.

In [ ]:
def extrair_janelas(df):
    """df com path, start_s, bucket -> (descritores (n,80), sequências (n,32,16)), na mesma ordem das linhas."""
    df = df.reset_index(drop=True); caminhos, inicios, buckets = df.path.to_numpy(), df.start_s.to_numpy(), df.bucket.to_numpy()
    def tarefa(item):
        caminho, linhas = item; onda = ler_audio_8k(RAIZ / caminho); saida = []
        for i in linhas:
            n = N_JANELA[buckets[i]]; a = int(round(inicios[i] * 8000)); seg = onda[a:a + n]
            if len(seg) != n: raise ValueError(f'janela fora do áudio: {caminho} @ {inicios[i]}')
            saida.append((i, seg))
        return saida
    janelas = {}
    with ThreadPoolExecutor(8) as ex:
        for saida in ex.map(tarefa, df.groupby('path').indices.items()):
            janelas.update(saida)
    F, S = np.zeros((len(df), 80), 'float32'), np.zeros((len(df), T_SEQ, 16), 'float32')
    for bucket in ('long', 'short'):
        ids = np.flatnonzero(buckets == bucket)
        for c in range(0, len(ids), 256):
            ix = ids[c:c + 256]
            db = espectrograma_mel_db(resample_poly(np.stack([janelas[i] for i in ix]), 6, 1, axis=1))
            F[ix], S[ix] = descrever(db), sequencia(db)
    return F, S

def media_por_chave(F, S, chaves, ordem):
    """Média de descritores/sequências das janelas que têm a mesma chave, na ordem dada."""
    pos = pd.Series(chaves).map({k: i for i, k in enumerate(ordem)}).to_numpy()
    n = np.bincount(pos, minlength=len(ordem)).astype(float)
    Fm, Sm = np.zeros((len(ordem), F.shape[1]), 'float32'), np.zeros((len(ordem),) + S.shape[1:], 'float32')
    np.add.at(Fm, pos, F); np.add.at(Sm, pos, S)
    return Fm / n[:, None], Sm / n[:, None, None]

def aparelho_funchal(caminho):
    """O aparelho é a taxa de amostragem escrita na pasta: '8khz', '44.1khz' ou '48khz'."""
    taxa = re.search(r'(\d+(?:\.\d+)?)\s*khz', caminho, re.I).group(1)
    return 'funchal:' + taxa + 'k'

def construir_tabela():
    # 1) corpus original: média das 3 janelas mais energéticas de cada grupo
    F, S = extrair_janelas(selecionadas)
    Fo, So = media_por_chave(F, S, selecionadas.group.to_numpy(), grupos_orig.group.tolist())
    # 2) Torres: tensores publicados (300 espectrogramas × 188 quadros × 32 faixas)
    bruto = np.asarray(json.loads(TORRES_JSON.read_text())['mspec'], 'float32'); assert bruto.shape == (300, 188, 32)
    Ft, St = descrever(bruto), sequencia(bruto)
    # 3) fontes externas: média das janelas de cada arquivo, arquivos em ordem alfabética
    ext = manifesto[manifesto.domain.isin(['figshare', 'mosquitosong', 'ufrgs'])].drop_duplicates(['path', 'start_s', 'bucket']).reset_index(drop=True)
    Fe_j, Se_j = extrair_janelas(ext); arquivos = sorted(ext.path.unique())
    Fe, Se = media_por_chave(Fe_j, Se_j, ext.path.to_numpy(), arquivos)
    meta = ext.groupby('path').agg(rotulo=('label', 'first'), dominio=('domain', 'first'), grupo=('group', 'first')).loc[arquivos]
    y_e = np.where(meta.rotulo.eq('Aedes aegypti'), 2, 1)
    src_e = meta.dominio.replace({'figshare': 'funchal'}).to_numpy()
    grp_e = np.array([aparelho_funchal(p) if d == 'figshare' else f'{d}:{g}' for p, d, g in zip(arquivos, meta.dominio, meta.grupo)])
    return dict(
        X=np.vstack([Fo, Ft, Fe]), S=np.concatenate([So, St, Se]),
        y=np.r_[grupos_orig.classe.to_numpy(), np.full(300, 2), y_e],
        src=np.r_[grupos_orig.dominio.to_numpy(), ['torres'] * 300, src_e].astype(str),
        grp=np.r_[grupos_orig.dominio.to_numpy() + ':' + grupos_orig.bloco.to_numpy(), [f'torres:{i // 10}' for i in range(300)], grp_e].astype(str))

arq_tabela = CACHE / 'tabela_atributos.npz'
if arq_tabela.exists() and not RECALCULAR_ATRIBUTOS:
    with np.load(arq_tabela) as z: tabela = {k: z[k] for k in z.files}
else:
    t0 = time.time(); tabela = construir_tabela(); np.savez_compressed(arq_tabela, **tabela)
    print(f'tabela construída em {(time.time() - t0) / 60:.1f} min')
X0, S0, y, fonte, grupo = tabela['X'], tabela['S'], tabela['y'], tabela['src'], tabela['grp']
assert np.isfinite(X0).all() and np.isfinite(S0).all()
print(X0.shape, '= gravações × atributos |', S0.shape, '= sequências para LSTM')
pd.crosstab(fonte, pd.Series(y).map(dict(enumerate(CLASSES))).to_numpy(), margins=True)

### O conjunto usado pelo modelo

Aplicamos a regra **"Aedes só de Torres e Funchal"**: as outras fontes entram só com o que **não** é Aedes (ruídos e outros mosquitos). Os Aedes das demais fontes ficam guardados para o teste da §17.

In [ ]:
usar = ((y == 2) & np.isin(fonte, FONTES_DE_AEDES)) | (y < 2)    # linhas que treinam e provam o modelo
idx_all = np.flatnonzero(usar)
contagem = {c: int((y[idx_all] == i).sum()) for i, c in enumerate(CLASSES)}
print(contagem, '| total', len(idx_all))
assert contagem == {'ruído': 426, 'outro mosquito': 3050, 'Aedes': 2167}, 'a base não bate com a apresentação'
print('Aedes guardados para o teste de fontes nunca vistas:', int(((y == 2) & ~usar).sum()))

### Como é o "espectro médio" de cada classe?

Média da energia em cada uma das 16 faixas. **Atenção:** parte da diferença vem do aparelho de cada fonte, não do mosquito. É por isso que normalizamos por fonte no próximo passo.

In [ ]:
for c, cor in zip([0, 1, 2], ['gray', 'tab:blue', 'tab:red']):
    plt.plot(CENTROS, X0[idx_all][y[idx_all] == c, :16].mean(0), 'o-', color=cor, label=CLASSES[c])
plt.xscale('log'); plt.xlabel('frequência (Hz)'); plt.ylabel('energia média (dB relativo)')
plt.legend(); plt.title('Espectro médio por classe (dados brutos)'); plt.show()

## 6. Preparo: três ajustes que valem mais que trocar de modelo

Na prática, a maior parte do trabalho de ML vai em dados e em provar que funciona. Os três ajustes abaixo foram os que mais mudaram o resultado:

**1. Normalizar por fonte.** Cada aparelho tem seu "sotaque" (microfone, ganho, ruído de fundo). Centralizamos cada atributo **dentro da própria fonte** (média 0, desvio 1): some o sotaque do microfone, fica o mosquito. As estatísticas vêm só das linhas que treinam o modelo.

**2. Pesos por fonte.** Uma fonte com 1.800 exemplos não pode "gritar" mais alto que uma com 20. Dentro de cada classe, cada fonte pesa igual.

**3. Dois estágios** (§9): primeiro *tem mosquito?*; depois, só entre mosquitos, *é Aedes?*. Um modelo para cada pergunta, porque as perguntas pedem sinais diferentes (tem energia de zumbido? esse zumbido é de Aedes?).

**Termos desta seção**

- **Normalização (z-score)**: subtrair a média e dividir pelo desvio padrão, deixando média 0 e desvio 1.
- **Peso de exemplo**: quanto cada exemplo conta no ajuste do modelo.
- **Estágio**: cada uma das duas perguntas da cascata: "tem mosquito?" e "é Aedes?".

In [ ]:
def estatisticas_por_fonte(A, fonte, usar):
    """Média e desvio de cada atributo, por fonte, usando só as linhas de `usar` (a fonte inteira entra no resultado)."""
    eixos = tuple(range(A.ndim - 1)); est = {}
    for s in np.unique(fonte):
        ref = usar & (fonte == s)
        if ref.any(): est[s] = (A[ref].mean(eixos), A[ref].std(eixos) + 1e-6)
    return est

def normalizar_por_fonte(A, fonte, est):
    An = A.copy()
    for s, (mu, sd) in est.items():
        k = fonte == s; An[k] = (A[k] - mu) / sd
    return An

def pesos_por_fonte(yy, ff):
    """Dentro de cada classe, cada fonte pesa igual: peso = 1 / (nº de exemplos da fonte) / (nº de fontes da classe)."""
    w = np.ones(len(yy))
    for c in np.unique(yy):
        mc = yy == c; fs = np.unique(ff[mc])
        for s in fs:
            k = mc & (ff == s); w[k] = 1 / k.sum() / len(fs)
    return w / w.mean()

EST = estatisticas_por_fonte(X0, fonte, usar)
Xn = normalizar_por_fonte(X0, fonte, EST)
Sn = normalizar_por_fonte(S0, fonte, estatisticas_por_fonte(S0, fonte, usar))

# o "sotaque": energia média da faixa de ~500 Hz, em cada fonte, antes e depois
fontes_uso = sorted(np.unique(fonte[usar])); faixa = int(np.argmin(abs(CENTROS - 500)))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=False)
for a, A, t in zip(ax, [X0, Xn], ['antes (dB relativo)', 'depois (média 0, desvio 1)']):
    a.boxplot([A[usar & (fonte == s), faixa] for s in fontes_uso], showfliers=False); a.set_xticks(range(1, len(fontes_uso) + 1), fontes_uso); a.set_title(f'Faixa de ~{CENTROS[faixa]:.0f} Hz — {t}')
    a.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

w_ex = pesos_por_fonte(y[idx_all], fonte[idx_all])
quadro_w = pd.DataFrame({'classe': pd.Series(y[idx_all]).map(dict(enumerate(CLASSES))).to_numpy(), 'fonte': fonte[idx_all], 'w': w_ex})
quadro_w.groupby(['classe', 'fonte']).agg(exemplos=('w', 'size'), peso_total=('w', 'sum')).round(0)

## 7. Não deixar o modelo "colar na prova"

**A regra de ouro.** Se o mesmo aparelho (ou a mesma gravação) aparece no treino e na prova, o modelo pode decorar o aparelho, e a nota fica inflada. Por isso a validação é por **grupo**: `StratifiedGroupKFold` divide em 5 dobras de modo que **um grupo inteiro fica de um lado só**, mantendo a proporção das classes. Repetimos 3 vezes com sorteios diferentes (sementes 11, 22, 33).

A célula abaixo mostra a diferença. Com dobras por linha (o jeito "comum"), quase todo grupo da prova já foi visto no treino. Com dobras por grupo, nenhum.

**Termos desta seção**

- **Validação cruzada**: dividir os dados em partes (dobras); treinar com algumas e provar nas outras, alternando até todas serem prova.
- **Dobra (fold)**: cada uma dessas partes.
- **Estratificada**: que mantém a proporção das classes em cada dobra.
- **Vazamento**: quando informação da prova escapa para o treino e infla o resultado.
- **assert**: verificação que interrompe o programa se a condição for falsa.

In [ ]:
from sklearn.model_selection import StratifiedKFold

def dobras(semente):
    """5 dobras por grupo sobre as linhas usadas; devolve índices globais (treino, prova)."""
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=semente).split(Xn[idx_all], y[idx_all], grupo[idx_all]):
        tr, te = idx_all[a], idx_all[b]
        assert not set(grupo[tr]) & set(grupo[te]), 'vazamento: o mesmo grupo no treino e na prova!'
        yield tr, te

def fracao_vista(split):
    vistos = [np.isin(grupo[te], grupo[tr]).mean() for tr, te in split]; return float(np.mean(vistos))

por_linha = ((idx_all[a], idx_all[b]) for a, b in StratifiedKFold(5, shuffle=True, random_state=11).split(Xn[idx_all], y[idx_all]))
print(f'dobras por linha : {fracao_vista(por_linha):.0%} das gravações da prova pertencem a um grupo que já estava no treino')
print(f'dobras por grupo : {fracao_vista(dobras(11)):.0%}  (o assert dentro de `dobras` garante)')

## 8. Como medir um detector

**Acurácia não basta.** Com classes desbalanceadas, "acertar 90%" pode significar só chutar a classe mais comum. Três perguntas, a partir da **matriz de confusão**:

* **Precisão** = VP / (VP + FP): quando o modelo apita "Aedes", com que frequência acerta?
* **Detecção** (*recall*) = VP / (VP + FN): de todos os Aedes que existiam, quantos ele achou?
* **AUROC**: sorteie um Aedes e um não-Aedes; qual a chance de o Aedes ganhar a nota maior? 0,5 = moeda, 1 = perfeito. Não depende do limiar de corte.

O modelo devolve uma **nota de Aedes** entre 0 e 1; quem escolhe o **limiar** de corte somos nós (§13).

**Termos desta seção**

- **Matriz de confusão**: tabela que cruza o que era de verdade com o que o modelo disse.
- **VP, FP, FN, VN**: verdadeiro positivo (Aedes certo), falso positivo (alarme falso), falso negativo (Aedes perdido), verdadeiro negativo (não-Aedes certo).
- **Precisão**: VP / (VP + FP): quando o modelo diz Aedes, quanto ele acerta.
- **Detecção (recall)**: VP / (VP + FN): dos Aedes que existiam, quantos ele achou.
- **AUROC**: área sob a curva ROC (§13): a chance de um Aedes sorteado ganhar nota maior que um não-Aedes sorteado. 0,5 = moeda, 1 = perfeito.
- **Nota (score)**: número de 0 a 1 que o modelo dá; quanto maior, mais ele acredita que é Aedes.
- **Limiar**: valor de corte da nota: acima dele, o modelo declara "Aedes".
- **Classes desbalanceadas**: quando uma classe tem muito mais exemplos que as outras.

In [ ]:
def metricas(y_verdade, nota_aedes, limiar=0.5):
    """Precisão, detecção e AUROC para Aedes (2) contra o resto, a partir da nota de Aedes."""
    ae = y_verdade == 2; chama = nota_aedes > limiar
    return dict(auroc=roc_auc_score(ae, nota_aedes),
                precisao=float(ae[chama].mean()) if chama.any() else 0.0,
                deteccao=float(chama[ae].mean()),
                outros_alarme=float(chama[y_verdade == 1].mean()),
                ruido_alarme=float(chama[y_verdade == 0].mean()))
print('definidas: metricas(y, nota, limiar)')

## 9. Os modelos

Todos recebem os mesmos 80 números (as redes recorrentes recebem a versão em sequência), a mesma receita (normalização, pesos, dois estágios) e a mesma prova. Só muda a forma de aprender.

**Árvores.** Uma *árvore de decisão* é uma sequência de perguntas sim/não sobre os atributos ("energia em ~500 Hz acima de X?"). Dois jeitos de juntar muitas árvores:
* **HistGB (boosting):** árvores pequenas em fila; cada nova árvore foca nos erros das anteriores.
* **Extra Trees (floresta):** centenas de árvores independentes com cortes sorteados; cada uma vota.

Árvores funcionam bem com atributos tabulares, poucos milhares de exemplos e sem GPU.

**Regressão logística.** Uma soma ponderada dos 80 números passada por uma curva em S. É o "modelo simples" de referência.

**SVM (RBF).** Procura a fronteira que separa as classes com a **maior margem**; só os pontos na beira da margem (os *vetores de suporte*) a definem. O kernel RBF mede a semelhança entre gravações e permite fronteiras curvas. Como mede distâncias, precisa de atributos na mesma escala: exatamente o que a normalização por fonte entrega.

**Redes neurais.** *MLP*: camadas de neurônios que somam os 80 números com pesos e passam por ReLU. *CNN*: um filtro pequeno desliza pelas 16 faixas procurando padrões ("pico + harmônico") em qualquer posição. *LSTM* e *Bi-LSTM*: leem os 32 instantes do espectrograma em ordem (a Bi-LSTM, nos dois sentidos). *DANN*: um MLP com uma segunda cabeça que tenta adivinhar a fonte; o treino pune o MLP quando ela acerta, para que ele esqueça o aparelho.

**CORAL + árvores.** Adaptação de domínio: "branqueia" cada fonte e a recolore com a covariância comum, antes das árvores.

**Termos desta seção**

- **Árvore de decisão**: sequência de perguntas sim/não sobre os atributos que leva a uma resposta.
- **Boosting (HistGB)**: árvores pequenas em fila, cada uma corrigindo os erros das anteriores.
- **Floresta (Extra Trees)**: centenas de árvores independentes com cortes sorteados; a resposta é a média dos votos.
- **Ensemble**: combinar vários modelos (aqui, tirando a média das notas).
- **Regressão logística**: soma ponderada dos atributos passada por uma curva em S que a transforma em probabilidade.
- **SVM (máquina de vetores de suporte)**: procura a fronteira entre as classes com a maior folga (margem).
- **Margem e vetores de suporte**: margem é a distância entre a fronteira e os exemplos mais próximos; esses exemplos mais próximos são os vetores de suporte.
- **Kernel RBF**: forma de medir a semelhança entre duas gravações que permite fronteiras curvas.
- **Rede neural**: modelo feito de camadas de "neurônios", cada um somando entradas com pesos.
- **Neurônio, peso e parâmetro**: neurônio é uma soma ponderada seguida de uma função; os pesos (parâmetros) são os números ajustados no treino.
- **ReLU**: função que zera valores negativos e mantém os positivos.
- **Dropout**: desligar neurônios ao acaso durante o treino, para a rede não decorar.
- **MLP**: rede de camadas de neurônios conectadas em sequência (perceptron multicamadas).
- **CNN e convolução**: rede em que um filtro pequeno desliza pelos dados procurando um padrão em qualquer posição.
- **LSTM e Bi-LSTM**: redes que leem uma sequência no tempo guardando memória do que veio antes; a Bi-LSTM lê nos dois sentidos.
- **DANN**: rede com uma segunda cabeça que tenta adivinhar a fonte; o treino pune a rede quando ela acerta, para que "esqueça" o aparelho.
- **CORAL e covariância**: covariância mede como os atributos variam juntos; o CORAL iguala a covariância de cada fonte à comum.
- **Época, lote e perda**: época é uma passada por todos os exemplos de treino; lote é o grupo de exemplos usado por vez; perda (loss) é o número de erro que o treino tenta reduzir.
- **Gradiente e otimizador (AdamW)**: o gradiente indica para onde mudar os pesos para errar menos; o otimizador é a regra que aplica a mudança.
- **Hiperparâmetro**: escolha feita antes do treino (número de árvores, taxa de aprendizado…), que não é ajustada pelos dados.
- **Probabilidade (predict_proba)**: a função que devolve a probabilidade de cada classe.

In [ ]:
# ---------- modelos clássicos ----------
def hgb(s):   # boosting: árvores pequenas em fila
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, l2_regularization=1.0,
                                          min_samples_leaf=12, random_state=s)
def floresta(s):   # Extra Trees: árvores independentes que votam
    return ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, n_jobs=8, random_state=s)
def svm(s):   # o modelo final
    return SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=s)
def logistica():
    return LogisticRegression(C=0.5, max_iter=5000)

# ---------- redes neurais ----------
def mlp():   # 80 -> 128 -> 64 -> 1
    return nn.Sequential(nn.Linear(80, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))

class CNN(nn.Module):   # entrada: 5 resumos (canais) × 16 faixas de frequência
    def __init__(s):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(5, 32, 3, padding=1), nn.ReLU(), nn.Conv1d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool1d(1))
        s.cabeca = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))
    def forward(s, x): return s.cabeca(s.conv(x.view(-1, 5, 16)))

class Recorrente(nn.Module):   # LSTM / Bi-LSTM: lê 32 instantes × 16 faixas
    def __init__(s, bidirecional):
        super().__init__(); s.rnn = nn.LSTM(16, 64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=bidirecional)
        s.saida = nn.Sequential(nn.Dropout(0.3), nn.Linear(128 if bidirecional else 64, 1))
    def forward(s, x): h, _ = s.rnn(x); return s.saida(h[:, -1])

class InverteGradiente(torch.autograd.Function):   # na volta, troca o sinal do gradiente: a rede aprende a "esquecer" a fonte
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None

FONTES = sorted(np.unique(fonte[usar])); ID_FONTE = np.array([FONTES.index(s) if s in FONTES else 0 for s in fonte])
class DANN(nn.Module):   # MLP + cabeça adversária que tenta adivinhar a fonte
    def __init__(s):
        super().__init__()
        s.corpo = nn.Sequential(nn.Linear(80, 128), nn.ReLU(), nn.Dropout(.3), nn.Linear(128, 64), nn.ReLU())
        s.nota = nn.Linear(64, 1); s.adversaria = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, len(FONTES)))
    def forward(s, x, lam=0.3): h = s.corpo(x); return s.nota(h), s.adversaria(InverteGradiente.apply(h, lam))

def ajustar_rede(tipo, A, yy, w, dominio, semente):
    """Treina a rede (AdamW, lotes de 128) e devolve uma função  nota = prever(B)."""
    torch.manual_seed(semente)
    rede = {'mlp': mlp, 'cnn': CNN, 'lstm': lambda: Recorrente(False), 'bilstm': lambda: Recorrente(True), 'dann': DANN}[tipo]().to(DISPOSITIVO)
    otim = torch.optim.AdamW(rede.parameters(), lr=1e-3, weight_decay=1e-3)
    At = torch.tensor(A, dtype=torch.float32, device=DISPOSITIVO); yt = torch.tensor(yy, dtype=torch.float32, device=DISPOSITIVO)
    wt = torch.tensor(w, dtype=torch.float32, device=DISPOSITIVO); dt = torch.tensor(dominio, device=DISPOSITIVO)
    gerador = torch.Generator().manual_seed(semente); bce = nn.functional.binary_cross_entropy_with_logits
    for _ in range(40 if tipo in ('lstm', 'bilstm') else 60):          # épocas: mostrar os exemplos várias vezes
        rede.train()
        for ix in torch.randperm(len(At), generator=gerador).split(128):
            ix = ix.to(DISPOSITIVO)
            if tipo == 'dann':
                nota, fonte_prevista = rede(At[ix])
                perda = (bce(nota.squeeze(1), yt[ix], reduction='none') * wt[ix]).mean() + nn.functional.cross_entropy(fonte_prevista, dt[ix])
            else:
                perda = (bce(rede(At[ix]).squeeze(1), yt[ix], reduction='none') * wt[ix]).mean()
            otim.zero_grad(); perda.backward(); nn.utils.clip_grad_norm_(rede.parameters(), 3); otim.step()
    rede.eval()
    def prever(B):
        with torch.no_grad():
            o = rede(torch.tensor(B, dtype=torch.float32, device=DISPOSITIVO)); o = o[0] if isinstance(o, tuple) else o
        return torch.sigmoid(o.squeeze(1)).cpu().numpy()
    return prever

print('parâmetros: MLP', sum(p.numel() for p in mlp().parameters()), '| CNN', sum(p.numel() for p in CNN().parameters()))

### CORAL e os dados de cada modelo

A maioria dos modelos usa os 80 atributos normalizados (`Xn`). A LSTM e a Bi-LSTM usam a sequência normalizada (`Sn`). O CORAL substitui a normalização por uma adaptação de domínio.

In [ ]:
def coral(A):
    """CORAL: branqueia cada fonte (remove a covariância dela) e recolore com a covariância comum a todas."""
    raiz = lambda C, e: (lambda w_, v: (v * np.clip(w_, 1e-6, None) ** e) @ v.T)(*np.linalg.eigh(C))
    centrado = np.concatenate([A[usar & (fonte == s)] - A[usar & (fonte == s)].mean(0) for s in FONTES])
    alvo = raiz(LedoitWolf().fit(centrado).covariance_, .5); An = A.copy()
    for s in FONTES:
        k = usar & (fonte == s); Z = A[k] - A[k].mean(0)
        An[k] = Z @ raiz(LedoitWolf().fit(Z).covariance_, -.5) @ alvo
    return An

DADOS = {'logistica': Xn, 'svm': Xn, 'mlp': Xn, 'cnn': Xn, 'dann': Xn, 'arvores': Xn, 'coral_arvores': coral(X0), 'lstm': Sn, 'bilstm': Sn}
NOMES = {'svm': ('SVM (RBF)', 'clássico'), 'dann': ('DANN', 'rede neural'), 'cnn': ('CNN', 'rede neural'), 'mlp': ('MLP', 'rede neural'),
         'bilstm': ('Bi-LSTM', 'rede neural'), 'arvores': ('Árvores (HistGB + ET)', 'clássico'), 'lstm': ('LSTM', 'rede neural'),
         'logistica': ('Regressão logística', 'clássico'), 'coral_arvores': ('CORAL + árvores', 'adaptação de domínio')}
print({k: v.shape for k, v in DADOS.items()})

### A receita completa em uma função: o modelo em dois estágios

* **Estágio 1** (*tem mosquito?*): treina com todas as linhas de treino; `y1 = 1` se for mosquito (qualquer espécie), `0` se for ruído.
* **Estágio 2** (*é Aedes?*): treina só com as linhas que são mosquito; `y2 = 1` se for Aedes.
* **Juntando** as duas notas como probabilidades: `ruído = 1 − p_m`, `outro mosquito = p_m·(1 − p_a)`, `Aedes = p_m·p_a`. A nota de Aedes só é alta se o modelo acha que **há mosquito e que é Aedes**.

Os pesos por fonte (§6) são recalculados dentro de cada estágio, só com as linhas de treino daquele estágio.

**Termos desta seção**

- **p_m e p_a**: p_m = probabilidade de haver mosquito; p_a = probabilidade de ser Aedes, dado que é mosquito.

In [ ]:
def nota_etapa(tipo, A, tr, yy, te, semente, etapa1):
    """Treina um estágio com as linhas `tr` (rótulos `yy`) e devolve a nota das linhas `te`."""
    w = pesos_por_fonte(yy, fonte[tr])
    if tipo == 'logistica': return logistica().fit(A[tr], yy, sample_weight=w).predict_proba(A[te])[:, 1]
    if tipo == 'svm':       return svm(semente).fit(A[tr], yy, sample_weight=w).predict_proba(A[te])[:, 1]
    if tipo in ('arvores', 'coral_arvores'):
        p = hgb(semente).fit(A[tr], yy, sample_weight=w).predict_proba(A[te])[:, 1]
        if etapa1: return p       # estágio 1: só boosting; estágio 2: média de boosting e floresta
        return (p + floresta(semente).fit(A[tr], yy, sample_weight=w).predict_proba(A[te])[:, 1]) / 2
    return ajustar_rede(tipo, A[tr], yy, w, ID_FONTE[tr], semente)(A[te])

def cascata(tipo, A, tr, te, semente):
    """Devolve (p_mosquito, p_aedes_dado_mosquito) para as linhas `te`, treinando só com `tr`."""
    y1 = (y[tr] > 0).astype(float); p_m = nota_etapa(tipo, A, tr, y1, te, semente, True)
    mos = tr[y[tr] > 0]; y2 = (y[mos] == 2).astype(float); p_a = nota_etapa(tipo, A, mos, y2, te, semente, False)
    return p_m, p_a

def ternaria(p_m, p_a):
    return np.c_[1 - p_m, p_m * (1 - p_a), p_m * p_a]      # [ruído, outro mosquito, Aedes]
print('definidas: nota_etapa, cascata, ternaria')

## 10. Qual modelo escolher? 9 modelos, a mesma prova

Mesmos dados, mesmos ajustes (normalização, pesos, dois estágios), validação por grupo, **média de 3 repetições** (sementes 11, 22, 33). Para cada modelo, cada gravação recebe uma nota de um modelo que **nunca a viu** (nem o grupo dela). Depois medimos AUROC, precisão e detecção (limiar 0,5) e a **AUROC Aedes × outros mosquitos nos mesmos aparelhos do Funchal** (§14).

> Esta é a etapa mais demorada. O resultado é guardado em `cache_notebook/ranking.json`; mude `RECALCULAR_RANKING` na §1 para refazer.

**Termos desta seção**

- **Repetição e média ± desvio**: repetimos a validação 3 vezes com sorteios diferentes; "0,86 ± 0,01" é a média e o quanto variou.
- **AUROC Aedes × Culex**: AUROC calculada só entre Aedes e outros mosquitos (Culex e Culiseta) gravados nos mesmos aparelhos (§14).
- **Ablação**: desligar uma peça do método para medir quanto ela valia.

In [ ]:
fun = fonte[idx_all] == 'funchal'

def avaliar_modelo(tipo):
    corridas = []
    for s in SEMENTES:
        nota = np.zeros(len(y))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(Xn[idx_all], y[idx_all], grupo[idx_all]):
            tr, te = idx_all[tr], idx_all[te]; assert not set(grupo[tr]) & set(grupo[te])
            p_m, p_a = cascata(tipo, DADOS[tipo], tr, te, s); nota[te] = p_m * p_a
        m = metricas(y[idx_all], nota[idx_all])
        m['auroc_aedes_x_culex'] = roc_auc_score(y[idx_all][fun] == 2, nota[idx_all][fun]); corridas.append(m)
    return {k: [round(float(np.mean([c[k] for c in corridas])), 3), round(float(np.std([c[k] for c in corridas])), 3)] for k in corridas[0]}

arq_rank = CACHE / 'ranking.json'
ranking = json.loads(arq_rank.read_text()) if arq_rank.exists() and not RECALCULAR_RANKING else {}
for tipo in NOMES:
    if tipo in ranking: continue
    t0 = time.time(); ranking[tipo] = avaliar_modelo(tipo); arq_rank.write_text(json.dumps(ranking, indent=1))
    print(f'{NOMES[tipo][0]:24s} AUROC {ranking[tipo]["auroc"][0]:.3f} | {(time.time() - t0) / 60:.1f} min', flush=True)

In [ ]:
ordem = sorted(ranking, key=lambda k: (-ranking[k]['auroc'][0], -ranking[k]['precisao'][0]))
tab = pd.DataFrame({NOMES[k][0]: {'tipo': NOMES[k][1], **{m: f'{ranking[k][m][0]:.2f} ± {ranking[k][m][1]:.2f}'
                     for m in ['auroc', 'precisao', 'deteccao', 'outros_alarme', 'ruido_alarme', 'auroc_aedes_x_culex']}} for k in ordem}).T
tab.columns = ['tipo', 'AUROC', 'precisão', 'detecção', 'outros→alarme', 'ruído→alarme', 'AUROC Aedes×Culex']; display(tab)

cores = {'clássico': '#4c8a4a', 'rede neural': '#3d5a98', 'adaptação de domínio': '#8a8577'}
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh([NOMES[k][0] for k in ordem], [ranking[k]['auroc'][0] for k in ordem], xerr=[ranking[k]['auroc'][1] for k in ordem],
        color=[cores[NOMES[k][1]] for k in ordem]); ax.axvline(.5, color='tab:red', ls='--'); ax.text(.505, len(ordem) - .6, 'acaso', color='tab:red')
ax.invert_yaxis(); ax.set_xlim(.4, 1); ax.set_xlabel('AUROC de Aedes (média de 3 repetições)'); ax.set_title('9 modelos, a mesma prova'); plt.tight_layout(); plt.show()

**Como ler o ranking.** Barras de erro mostram o desvio entre as 3 repetições: diferenças menores que elas não são diferenças reais. Quando vários modelos empatam em AUROC, o desempate vem de precisão, de alarmes falsos e da prova Aedes × Culex. O modelo final é a **SVM**: está entre os melhores em AUROC, tem a maior precisão e o melhor resultado na prova mais justa, treina em segundos e não precisa de GPU.

### Quanto valeu cada ajuste? (ablação)

**Ablação** é desligar uma peça do método para medir quanto ela valia. Aqui usamos as árvores (HistGB) em três versões, cada uma acrescentando um ajuste da §6 ao anterior:

1. **sem ajustes**: um único modelo de 3 classes, atributos como saíram do espectrograma;
2. **+ normalização por fonte**;
3. **+ dois estágios** (com a floresta no segundo estágio).

Os pesos por fonte estão ligados nas três versões. Mesma prova da §10: validação por grupo, média de 3 repetições.

In [ ]:
def nota_tres_classes(A, tr, te, semente):
    """Um modelo só, com 3 classes (sem dois estágios); a nota de Aedes é a probabilidade da classe 2."""
    return hgb(semente).fit(A[tr], y[tr], sample_weight=pesos_por_fonte(y[tr], fonte[tr])).predict_proba(A[te])[:, 2]

VARIANTES = {'sem ajustes': lambda tr, te, s: nota_tres_classes(X0, tr, te, s),
             '+ normalização': lambda tr, te, s: nota_tres_classes(Xn, tr, te, s),
             '+ dois estágios': lambda tr, te, s: np.prod(cascata('arvores', Xn, tr, te, s), axis=0)}
arq_abl = CACHE / 'ablacao.json'
ablacao = json.loads(arq_abl.read_text()) if arq_abl.exists() and not RECALCULAR_RANKING else {}
for nome, funcao in VARIANTES.items():
    if nome in ablacao: continue
    corridas = []
    for s in SEMENTES:
        nota = np.zeros(len(y))
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(Xn[idx_all], y[idx_all], grupo[idx_all]):
            tr, te = idx_all[tr], idx_all[te]; assert not set(grupo[tr]) & set(grupo[te]); nota[te] = funcao(tr, te, s)
        m = metricas(y[idx_all], nota[idx_all]); m['auroc_aedes_x_culex'] = roc_auc_score(y[idx_all][fun] == 2, nota[idx_all][fun]); corridas.append(m)
    ablacao[nome] = {k: round(float(np.mean([c[k] for c in corridas])), 3) for k in corridas[0]}
    arq_abl.write_text(json.dumps(ablacao, indent=1))
display(pd.DataFrame(ablacao).T[['precisao', 'outros_alarme', 'auroc', 'auroc_aedes_x_culex']].rename(
    columns={'precisao': 'precisão dos alertas', 'outros_alarme': 'outros mosquitos → alarme', 'auroc': 'AUROC', 'auroc_aedes_x_culex': 'AUROC Aedes×Culex'}))
fig, ax = plt.subplots(figsize=(8, 3.4)); larg = .25
for i, (rot, ch) in enumerate([('precisão dos alertas', 'precisao'), ('outros mosquitos → alarme', 'outros_alarme'), ('AUROC', 'auroc')]):
    ax.bar(np.arange(3) + (i - 1) * larg, [ablacao[v][ch] for v in VARIANTES], larg, label=rot)
ax.set_xticks(range(3), list(VARIANTES)); ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.set_title('O que cada ajuste mudou'); plt.tight_layout(); plt.show()

## 11. O modelo final: SVM em dois estágios

**O que faz:** roda a validação por grupo (5 dobras, semente 11) com a SVM em dois estágios, **escrita passo a passo** para mostrar a receita. Cada gravação recebe uma nota de um modelo que nunca a viu. Em cada dobra também medimos a **importância de cada faixa de frequência** (§15) embaralhando uma faixa por vez nos exemplos de prova.

**Por quê SVM:** ver o fim da §10. O resultado é guardado em `cache_notebook/modelo_final.npz`.

In [ ]:
SEMENTE_FINAL = 11
def colunas_da_faixa(b): return [b + 16 * k for k in range(5)]    # média, desvio, p10, p50, p90 da faixa b

def validar_modelo_final():
    P3 = np.zeros((len(y), 3)); perm = np.zeros((16, len(y))); sorteio = np.random.default_rng(0)
    for tr, te in dobras(SEMENTE_FINAL):
        # estágio 1: tem mosquito?
        y1 = (y[tr] > 0).astype(int)
        svm_mosquito = svm(SEMENTE_FINAL).fit(Xn[tr], y1, sample_weight=pesos_por_fonte(y1, fonte[tr]))
        # estágio 2: entre os mosquitos, é Aedes?
        mos = tr[y[tr] > 0]; y2 = (y[mos] == 2).astype(int)
        svm_aedes = svm(SEMENTE_FINAL).fit(Xn[mos], y2, sample_weight=pesos_por_fonte(y2, fonte[mos]))
        prever = lambda Z: ternaria(svm_mosquito.predict_proba(Z)[:, 1], svm_aedes.predict_proba(Z)[:, 1])
        P3[te] = prever(Xn[te])
        for b in range(16):                       # importância por permutação: embaralha uma faixa e mede o estrago
            Z = Xn[te].copy(); c = colunas_da_faixa(b); Z[:, c] = Z[sorteio.permutation(len(te))][:, c]
            perm[b, te] = prever(Z)[:, 2]
    return P3, perm

arq_final = CACHE / 'modelo_final.npz'
if arq_final.exists():
    with np.load(arq_final) as z: P3, perm = z['P3'], z['perm']
else:
    t0 = time.time(); P3, perm = validar_modelo_final(); np.savez_compressed(arq_final, P3=P3, perm=perm)
    print(f'validação do modelo final em {(time.time() - t0) / 60:.1f} min')
yt, pa = y[idx_all], P3[idx_all, 2]; ae = yt == 2
print('pronto: cada gravação recebeu uma previsão de um modelo que nunca viu o grupo dela')

## 12. A matriz de confusão

Linhas = o que era de verdade; colunas = o que o modelo disse. A **diagonal** são os acertos. Cada gravação foi prevista por um modelo que nunca a viu; o limiar de Aedes é 0,5.

In [ ]:
LIMIAR = 0.5
pred = np.where(pa > LIMIAR, 2, P3[idx_all, :2].argmax(1))
cm = confusion_matrix(yt, pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(cmap='Blues', colorbar=False, values_format='d')
disp.ax_.set_xlabel('o modelo disse'); disp.ax_.set_ylabel('era de verdade'); plt.title(f'Matriz de confusão (limiar {LIMIAR}) — SVM'); plt.show()

vp, fp = cm[2, 2], cm[0, 2] + cm[1, 2]
print(f'{len(yt):,} gravações: {cm[2].sum():,} Aedes, {cm[1].sum():,} outros mosquitos, {cm[0].sum():,} ruídos')
print(f'Aedes certos: {cm[2, 2]:,} | Aedes chamados de "outro mosquito": {cm[2, 1]:,} | de "ruído": {cm[2, 0]:,}')
print(f'alarmes falsos: {cm[1, 2]:,} de {cm[1].sum():,} outros mosquitos ({cm[1, 2] / cm[1].sum():.0%}) e {cm[0, 2]:,} de {cm[0].sum():,} ruídos ({cm[0, 2] / cm[0].sum():.0%})')
print(f'precisão  = {vp:,} / {vp + fp:,} = {vp / (vp + fp):.0%}   (quando diz Aedes, acerta)')
print(f'detecção  = {vp:,} / {cm[2].sum():,} = {vp / cm[2].sum():.0%}   (dos Aedes, quantos achou)')

## 13. Curva ROC e a escolha do limiar

**Curva ROC.** Cada ponto é um limiar diferente: para cima = mais Aedes encontrados; para a direita = mais alarmes falsos. A diagonal é o chute (moeda). A **AUROC** é a área sob a curva.

**Limiar.** O modelo não diz "sim/não": dá uma nota de 0 a 1. **Onde cortar é uma decisão de projeto**: limiar baixo = vigilância, não perder Aedes (mais detecção, mais alarmes falsos); limiar alto = poucos alarmes.

**Termos desta seção**

- **Curva ROC**: gráfico de taxa de verdadeiros positivos (Aedes achados) contra taxa de falsos positivos (alarmes falsos), para todos os limiares.
- **Histograma**: gráfico de barras que conta quantos exemplos caem em cada faixa de valores.
- **Otimista**: resultado que parece melhor do que será em dados novos.

In [ ]:
fpr, tpr, _ = roc_curve(ae, pa)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(fpr, tpr, color='tab:red', lw=2, label=f'modelo (AUROC = {roc_auc_score(ae, pa):.2f})'); ax[0].plot([0, 1], [0, 1], '--', color='gray', label='moeda (0,5)')
ax[0].set_xlabel('alarmes falsos (taxa de falsos positivos)'); ax[0].set_ylabel('Aedes detectados (taxa de verdadeiros positivos)'); ax[0].legend(); ax[0].set_title('Curva ROC')

bins = np.linspace(0, 1, 21)
ax[1].hist(pa[ae], bins, alpha=.6, color='tab:red', label='Aedes', density=True); ax[1].hist(pa[yt == 1], bins, alpha=.6, color='tab:blue', label='outros mosquitos', density=True)
ax[1].hist(pa[yt == 0], bins, alpha=.6, color='gray', label='ruído', density=True)
ax[1].axvline(LIMIAR, color='k', ls='--'); ax[1].set_xlabel('nota de Aedes'); ax[1].legend(); ax[1].set_title('Distribuição das notas')

ts = np.round(np.arange(.05, .96, .05), 2)
linhas = [dict(limiar=t, **metricas(yt, pa, t)) for t in ts]
ax[2].plot(ts, [l['precisao'] for l in linhas], 'o-', label='precisão'); ax[2].plot(ts, [l['deteccao'] for l in linhas], 'o-', label='detecção (recall)')
ax[2].plot(ts, [l['outros_alarme'] for l in linhas], 'o-', label='outros mosquitos → alarme'); ax[2].set_xlabel('limiar'); ax[2].legend(); ax[2].set_title('O compromisso do limiar')
plt.tight_layout(); plt.show()
tab_limiar = pd.DataFrame(linhas).set_index('limiar').loc[[.2, .3, .4, .5, .6, .7]].round(2); display(tab_limiar[['precisao', 'deteccao', 'outros_alarme', 'ruido_alarme']])

### Escolher o limiar sem "colar"

Se escolhermos o limiar olhando as mesmas notas em que depois medimos o resultado, o resultado fica otimista, o mesmo erro da §7. Aqui a regra de projeto é: **"a maior detecção que mantém os alarmes falsos de outros mosquitos em até 10%"**. O limiar é escolhido em **metade dos grupos** (validação) e medido na **outra metade** (teste), sem que um grupo apareça nas duas.

In [ ]:
def escolher_limiar(nota, yy, max_alarme_outros=0.10):
    """Menor limiar (= maior detecção) que respeita o limite de alarmes falsos entre outros mosquitos."""
    candidatos = np.round(np.arange(.05, .96, .01), 2)
    validos = [t for t in candidatos if (nota[yy == 1] > t).mean() <= max_alarme_outros]
    return float(min(validos)) if validos else 0.95

val, tst = next(StratifiedGroupKFold(2, shuffle=True, random_state=5).split(Xn[idx_all], yt, grupo[idx_all]))
t_val = escolher_limiar(pa[val], yt[val])
print(f'limiar escolhido na metade de validação: {t_val:.2f}  (a regra fixa era {LIMIAR})')
for nome, t in [('limiar fixo 0,5', LIMIAR), (f'limiar escolhido {t_val:.2f}', t_val)]:
    m = metricas(yt[tst], pa[tst], t); print(f'{nome:24s} na metade de teste: detecção {m["deteccao"]:.0%} | precisão {m["precisao"]:.0%} | outros→alarme {m["outros_alarme"]:.0%}')

### Resumo do modelo final

Os números que aparecem nos slides de resultado, calculados de uma vez. **Cuidado ao ler:** o modelo é **cauteloso**: prefere ficar quieto a gritar "Aedes" sem ter certeza. Por isso a precisão é alta e a detecção, moderada.

In [ ]:
m50, m20 = metricas(yt, pa, .5), metricas(yt, pa, .2)
print(f'AUROC Aedes                       : {m50["auroc"]:.2f}')
print(f'limiar 0,5 → precisão {m50["precisao"]:.0%} · detecção {m50["deteccao"]:.0%} · {m50["outros_alarme"]:.0%} dos outros mosquitos e {m50["ruido_alarme"]:.0%} dos ruídos viram alarme')
print(f'limiar 0,2 → precisão {m20["precisao"]:.0%} · detecção {m20["deteccao"]:.0%} · {m20["outros_alarme"]:.0%} dos outros mosquitos e {m20["ruido_alarme"]:.0%} dos ruídos viram alarme')

### Nota: o resultado depende de como os aparelhos são agrupados

Só há **três aparelhos** de Funchal, e cada um é um grupo inteiro: qual aparelho cai em qual dobra muda bastante o resultado. A célula abaixo roda a mesma SVM (3 repetições) com dois agrupamentos: o **correto** (8, 44,1 e 48 kHz separados) e o **antigo**, em que 8 e 48 kHz ficavam juntos por um erro de comparação de texto. A diferença mostra o quanto a conclusão depende dessa escolha, e é uma razão para coletar mais aparelhos (§19).

In [ ]:
def svm_tres_repeticoes(grupos):
    corridas = []
    for s in SEMENTES:
        nota = np.zeros(len(y))
        for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=s).split(Xn[idx_all], y[idx_all], grupos[idx_all]):
            tr, te = idx_all[a], idx_all[b]; assert not set(grupos[tr]) & set(grupos[te])
            p_m, p_a = cascata('svm', Xn, tr, te, s); nota[te] = p_m * p_a
        m = metricas(y[idx_all], nota[idx_all]); m['auroc_aedes_x_culex'] = roc_auc_score(y[idx_all][fun] == 2, nota[idx_all][fun]); corridas.append(m)
    return {k: round(float(np.mean([c[k] for c in corridas])), 3) for k in corridas[0]}

grupo_antigo = np.where(fonte == 'funchal', np.where(np.char.find(grupo, '44.1k') >= 0, 'funchal:44.1k', 'funchal:8k+48k'), grupo)
arq_sens = CACHE / 'sensibilidade_grupos.json'
if arq_sens.exists(): sens = json.loads(arq_sens.read_text())
else:
    sens = {'aparelhos separados (correto)': svm_tres_repeticoes(grupo), '8 kHz e 48 kHz juntos (antigo)': svm_tres_repeticoes(grupo_antigo)}
    arq_sens.write_text(json.dumps(sens, indent=1))
pd.DataFrame(sens).T[['auroc', 'precisao', 'deteccao', 'outros_alarme', 'auroc_aedes_x_culex']]

## 14. A prova mais justa: Aedes × outros mosquitos no mesmo aparelho

Um modelo pode "acertar" só por reconhecer **o aparelho**: se todo o Aedes veio de um microfone e todo o resto de outro, ele aprende o microfone. A prova que fecha essa porta usa o **Funchal**: Aedes e outros mosquitos (Culex e Culiseta) foram gravados **nos mesmos aparelhos**. Como cada aparelho é um grupo, na prova um aparelho inteiro fica fora do treino, e a pergunta é: o modelo separa as espécies? O "sotaque" do microfone não ajuda, porque é igual para as duas; para acertar, o modelo precisa ouvir o mosquito.

In [ ]:
f = fonte[idx_all] == 'funchal'
print(f'AUROC Aedes × outros mosquitos, só Funchal, aparelho fora do treino: {roc_auc_score(ae[f], pa[f]):.2f}')
dispositivos = grupo[idx_all][f]
pd.DataFrame([dict(aparelho=d, aedes=int(((dispositivos == d) & ae[f]).sum()), outros=int(((dispositivos == d) & ~ae[f]).sum()),
                   auroc=roc_auc_score(ae[f][dispositivos == d], pa[f][dispositivos == d]) if len(set(ae[f][dispositivos == d])) == 2 else np.nan)
              for d in sorted(set(dispositivos))]).round(2)

**O que ainda falta.** O modelo só viu Aedes de **dois** datasets. Aedes gravados em outros aparelhos ainda não são reconhecidos (§17): o próximo passo é coletar mais fontes.

## 15. O que o modelo "ouve"? Importância por permutação

**As métricas dizem quanto o modelo acerta; a interpretabilidade pergunta em que ele se baseia.**

Para cada uma das 16 faixas, **embaralhamos** os 5 resumos daquela faixa entre as gravações da prova e medimos **quanto o AUROC cai**. Se embaralhar uma faixa derruba o AUROC, o modelo depende dela; se nada acontece, ela é irrelevante para ele.

**Termos desta seção**

- **Interpretabilidade**: descobrir em que o modelo se baseia para decidir.
- **Importância por permutação**: embaralhar uma parte dos dados e medir quanto o resultado piora.

In [ ]:
base = roc_auc_score(ae, pa)
quedas = np.array([base - roc_auc_score(ae, perm[b, idx_all]) for b in range(16)])
top3 = np.argsort(quedas)[-3:]
plt.figure(figsize=(10, 3.6))
plt.bar([f'{c:.0f}' for c in CENTROS], quedas, color=['tab:red' if b in top3 else 'tab:blue' for b in range(16)])
plt.xticks(rotation=45); plt.xlabel('centro da faixa (Hz)'); plt.ylabel('queda no AUROC'); plt.title('Importância de cada faixa de frequência (SVM)'); plt.show()
print('3 faixas com maior queda (Hz):', sorted(CENTROS[top3].round().astype(int).tolist()))
print('queda média nas faixas abaixo de 300 Hz:', round(float(quedas[CENTROS < 300].mean()), 4), '| entre 600 e 1600 Hz:', round(float(quedas[(CENTROS >= 600) & (CENTROS <= 1600)].mean()), 4))

## 16. E o aparelho? O teste de origem

Normalizar por fonte deveria apagar o "sotaque". Dá para conferir: treinamos um classificador **linear** para adivinhar **de qual fonte veio** cada **outro mosquito** (aqui não há Aedes, só mosquitos de várias coleções). Se ele acerta muito, o aparelho ainda é visível nos atributos. O acaso é `1 / nº de fontes`. Fazemos o teste **antes** e **depois** da normalização.

**Termos desta seção**

- **Classificador linear e sonda (probe)**: classificador linear separa classes com uma reta/plano; usado como sonda, serve para testar se uma informação (a fonte) ainda está nos atributos.
- **Acurácia balanceada**: média da taxa de acerto de cada classe: não é inflada pela classe maior.
- **Acaso**: o resultado de chutar: com 4 fontes, acertar 25% por sorte.

In [ ]:
om = usar & (y == 1); S_om = fonte[om]; fontes_om = sorted(set(S_om)); g_om = grupo[om]
def sondar_origem(A):
    pr = np.empty(len(S_om), object)
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=7).split(A, S_om, g_om):
        pr[b] = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight='balanced')).fit(A[a], S_om[a]).predict(A[b])
    return balanced_accuracy_score(S_om, pr)
print(f'fontes (só outros mosquitos): {fontes_om} | acaso = {1 / len(fontes_om):.0%}')
print(f'adivinha a fonte ANTES da normalização : {sondar_origem(X0[om]):.0%}')
print(f'adivinha a fonte DEPOIS da normalização: {sondar_origem(Xn[om]):.0%}')

## 17. Aedes de fontes nunca vistas

Os Aedes de Dryad, HumBug, MosquitoSong e UFRGS **não** entraram no treino nem na prova. Aqui eles viram o teste mais duro: treinamos o modelo final com **todos** os dados usados e perguntamos se ele reconhece Aedes gravados em aparelhos e coleções que nunca viu.

Cada fonte é normalizada com as estatísticas dela mesma (calculadas só com as suas gravações que **não** são Aedes), o mesmo procedimento do treino. Comparamos as notas dessas gravações com as notas **da prova** dos não-Aedes (§11).

In [ ]:
tr_all = idx_all
y1 = (y[tr_all] > 0).astype(int); sv_m = svm(SEMENTE_FINAL).fit(Xn[tr_all], y1, sample_weight=pesos_por_fonte(y1, fonte[tr_all]))
mos_all = tr_all[y[tr_all] > 0]; y2 = (y[mos_all] == 2).astype(int); sv_a = svm(SEMENTE_FINAL).fit(Xn[mos_all], y2, sample_weight=pesos_por_fonte(y2, fonte[mos_all]))
modelo_final = lambda Z: ternaria(sv_m.predict_proba(Z)[:, 1], sv_a.predict_proba(Z)[:, 1])

novos = (y == 2) & ~usar
nota_novos = modelo_final(Xn[novos])[:, 2]
nota_neg = pa[yt < 2]
linhas = []
for s in sorted(set(fonte[novos])):
    k = fonte[novos] == s
    linhas.append(dict(fonte=s, aedes=int(k.sum()), reconhecidos_limiar_05=f'{int((nota_novos[k] > .5).sum())}/{int(k.sum())}',
                       nota_mediana=round(float(np.median(nota_novos[k])), 2)))
display(pd.DataFrame(linhas))
auc_novos = roc_auc_score(np.r_[np.ones(len(nota_novos)), np.zeros(len(nota_neg))], np.r_[nota_novos, nota_neg])
print(f'AUROC (Aedes de fontes novas × não-Aedes da prova): {auc_novos:.2f}   (0,5 = acaso; abaixo de 0,5 = pior que o acaso)')
print(f'nota mediana dos não-Aedes da prova: {np.median(nota_neg):.2f}')

**O que isto diz.** Se a AUROC fica perto de 0,5 ou abaixo, o modelo **não** reconhece Aedes de aparelhos novos; os bons números das seções anteriores vêm de reconhecer as **duas fontes** de Aedes (Torres e Funchal) e de separá-las dos negativos. Este é o limite mais importante do trabalho e é a razão da lista de próximos passos (§19).

## 18. Aplicar o modelo a um WAV

**O que faz:** transforma um arquivo de áudio em uma resposta: as três probabilidades `[ruído, outro mosquito, Aedes]`.

**Como:** corta o áudio em janelas de 0,992 s, extrai os 80 números de cada uma (a mesma função da §4), tira a média (como fizemos com cada arquivo das fontes externas) e passa pelo modelo final.

**Atenção, a normalização.** O treino normaliza **por fonte**. Um WAV novo não tem fonte conhecida, então você escolhe `fonte_referencia`: as estatísticas de qual fonte de treino usar. Isso **não está validado** para aparelhos novos (ver §17): trate a resposta como uma estimativa, não como um diagnóstico.

**Termos desta seção**

- **Inferência**: usar o modelo já treinado para responder sobre uma gravação nova.

In [ ]:
def prever_wav(caminho, fonte_referencia='funchal', max_janelas=60):
    """Devolve {'ruído': p, 'outro mosquito': p, 'Aedes': p} para um arquivo de áudio."""
    onda = ler_audio_8k(caminho); n = N_JANELA['long']
    pedacos = [onda[i:i + n] for i in range(0, len(onda) - n + 1, 8000)][:max_janelas]      # janelas de 0,992 s, 1 por segundo
    if not pedacos: raise ValueError('o áudio precisa ter pelo menos 0,992 s')
    descritores = np.concatenate([descrever(atributos_da_janela(p)) for p in pedacos]).mean(0, keepdims=True)
    mu, sd = EST[fonte_referencia]
    p = modelo_final((descritores - mu) / sd)[0]
    return dict(zip(CLASSES, p.round(3)))

# exemplos (todos são arquivos do próprio projeto; o primeiro é de uma fonte de treino, o segundo de uma fonte que o modelo nunca viu)
def primeiro_arquivo(dominio, classe):
    return manifesto[(manifesto.domain == dominio) & (manifesto.classe == classe) & (manifesto.bucket == 'long')].path.iloc[0]
for dominio, classe in [('dryad', 2), ('dryad', 1), ('esc50', 0)]:
    caminho = primeiro_arquivo(dominio, classe)
    print(f'{dominio:6s} · era {CLASSES[classe]:15s} →', prever_wav(RAIZ / caminho, fonte_referencia=dominio))

## 19. Limites e próximos passos

* **Retrospectivo.** Todos os dados já eram conhecidos do projeto; não é um teste cego.
* **Dois datasets de Aedes.** A detecção reflete Torres e Funchal; Aedes de outros aparelhos ainda não são reconhecidos (§17).
* **Grupos não são indivíduos.** Torres não identifica indivíduos; 10 espectrogramas consecutivos formam um grupo. Os grupos protegem contra vazamento por gravação e aparelho, não provam independência biológica.
* **Ruído de ambientes reais.** Os ruídos vêm do ESC-50 e de gravações de apoio, não de armadilhas em campo.

**Receita para um teste de verdade**

1. Controles pareados: Aedes e outras espécies gravados no **mesmo aparelho, setup e dia**, em vários aparelhos e locais.
2. Ruído *in situ* dos ambientes reais.
3. Mais fontes de Aedes.
4. Um **teste cego único**, numa campanha nova, com o protocolo congelado antes da coleta.

## Referências

**Dados e trabalhos sobre mosquitos.** Mukundarajan H. et al. *Using mobile phones as acoustic sensors for high-throughput mosquito surveillance.* eLife 6:e27854, 2017 (dataset Abuzz e vídeos, CC BY 4.0). Kiskin I. et al. *HumBugDB.* NeurIPS Datasets and Benchmarks, 2021. Sinka M.E. et al. *HumBug.* Methods Ecol. Evol. 12, 2021. Torres I. et al. *Dengue-Infected Mosquito Detection with Uncertainty Evaluation based on Monte Carlo Dropout.* IEEE Latin America Trans. 23(2), 2025. Vasconcelos D. et al. *An annotated dataset of bioacoustic sensing and features of mosquitoes.* Scientific Data 7:382, 2020 (Funchal). Supratak A. et al. *MosquitoSong+.* PLOS ONE 19, 2024. Paim K.O. et al. *Acoustic Identification of Ae. aegypti Mosquitoes using Smartphone Apps and Residual CNNs.* arXiv:2306.10091, 2023 (UFRGS). Piczak K.J. *ESC: Dataset for Environmental Sound Classification.* ACM Multimedia, 2015.

**Métodos.** Cortes C., Vapnik V. *Support-vector networks.* Machine Learning 20, 1995 (SVM). Friedman J.H. *Greedy function approximation.* Ann. Statist. 29(5), 2001; Ke G. et al. *LightGBM.* NeurIPS, 2017 (HistGB). Geurts P. et al. *Extremely randomized trees.* Machine Learning 63, 2006. Hochreiter S., Schmidhuber J. *Long short-term memory.* Neural Computation 9(8), 1997. Ganin Y. et al. *Domain-adversarial training of neural networks.* JMLR 17, 2016 (DANN). Sun B. et al. *Return of frustratingly easy domain adaptation.* AAAI, 2016 (CORAL). Breiman L. *Random forests.* Machine Learning 45, 2001 (importância por permutação). Pedregosa F. et al. *Scikit-learn.* JMLR 12, 2011; Paszke A. et al. *PyTorch.* NeurIPS, 2019.

**Termos desta seção**

- **In situ**: no local real de uso (por exemplo, numa armadilha em campo).

## Glossário

Todos os termos definidos no caderno, em ordem alfabética (cada um foi definido na seção em que aparece pela primeira vez).

- **Ablação**: desligar uma peça do método para medir quanto ela valia.
- **Acaso**: o resultado de chutar: com 4 fontes, acertar 25% por sorte.
- **Acurácia balanceada**: média da taxa de acerto de cada classe: não é inflada pela classe maior.
- **Aedes aegypti**: o mosquito que transmite dengue, zika e chikungunya; é o alvo do detector.
- **Amostra**: um valor numérico do sinal de áudio em um instante.
- **Aparelho**: o equipamento que gravou (celular, gravador, microfone). Cada um tem o seu "sotaque".
- **Aprendizado de máquina (ML)**: programas que aprendem padrões a partir de exemplos, em vez de seguir regras escritas à mão.
- **Árvore de decisão**: sequência de perguntas sim/não sobre os atributos que leva a uma resposta.
- **assert**: verificação que interrompe o programa se a condição for falsa.
- **Atributo (feature)**: cada número que descreve uma gravação para o modelo.
- **AUROC**: área sob a curva ROC (§13): a chance de um Aedes sorteado ganhar nota maior que um não-Aedes sorteado. 0,5 = moeda, 1 = perfeito.
- **AUROC Aedes × Culex**: AUROC calculada só entre Aedes e outros mosquitos (Culex e Culiseta) gravados nos mesmos aparelhos (§14).
- **Banco de filtros e faixa**: conjunto de filtros triangulares; cada um soma a energia de uma faixa de frequências.
- **Biblioteca**: conjunto de funções prontas (NumPy para números, scikit-learn para modelos, PyTorch para redes neurais).
- **Boosting (HistGB)**: árvores pequenas em fila, cada uma corrigindo os erros das anteriores.
- **Cache**: cópia guardada de um resultado caro de calcular, para não recalcular toda vez.
- **Classe**: cada resposta possível do detector (aqui: ruído, outro mosquito, Aedes).
- **Classes desbalanceadas**: quando uma classe tem muito mais exemplos que as outras.
- **Classificador linear e sonda (probe)**: classificador linear separa classes com uma reta/plano; usado como sonda, serve para testar se uma informação (a fonte) ainda está nos atributos.
- **CNN e convolução**: rede em que um filtro pequeno desliza pelos dados procurando um padrão em qualquer posição.
- **CORAL e covariância**: covariância mede como os atributos variam juntos; o CORAL iguala a covariância de cada fonte à comum.
- **CPU e GPU**: CPU é o processador comum; GPU é a placa gráfica, que acelera o treino de redes neurais.
- **Curva ROC**: gráfico de taxa de verdadeiros positivos (Aedes achados) contra taxa de falsos positivos (alarmes falsos), para todos os limiares.
- **DANN**: rede com uma segunda cabeça que tenta adivinhar a fonte; o treino pune a rede quando ela acerta, para que "esqueça" o aparelho.
- **Dataset (base de dados)**: coleção de gravações com a resposta correta conhecida de cada uma.
- **Decibel (dB)**: escala logarítmica de energia: 10·log10(potência). Valores relativos ao pico são sempre ≤ 0 dB.
- **Desvio padrão**: o quanto os valores variam em torno da média.
- **Detecção (recall)**: VP / (VP + FN): dos Aedes que existiam, quantos ele achou.
- **Detector ternário**: classificador com três respostas possíveis: ruído, outro mosquito ou Aedes.
- **Dobra (fold)**: cada uma dessas partes.
- **Dropout**: desligar neurônios ao acaso durante o treino, para a rede não decorar.
- **Duplicata e bloco**: gravações com o mesmo som em arquivos diferentes. Um bloco junta todos os grupos que repetem o mesmo áudio.
- **Ensemble**: combinar vários modelos (aqui, tirando a média das notas).
- **Época, lote e perda**: época é uma passada por todos os exemplos de treino; lote é o grupo de exemplos usado por vez; perda (loss) é o número de erro que o treino tenta reduzir.
- **Escala mel**: escala de frequências que imita o ouvido: faixas estreitas nos graves e largas nos agudos.
- **Espectro**: gráfico da energia do sinal em cada frequência.
- **Espectrograma**: imagem da energia por frequência (vertical) ao longo do tempo (horizontal).
- **Estágio**: cada uma das duas perguntas da cascata: "tem mosquito?" e "é Aedes?".
- **Estratificada**: que mantém a proporção das classes em cada dobra.
- **ffmpeg**: programa que decodifica formatos de áudio que a biblioteca principal não abre (como .m4a).
- **FFT (transformada de Fourier)**: conta que converte um trecho de som no seu espectro.
- **FFT de 1024 e avanço de 256**: a janelinha tem 1024 amostras e anda 256 amostras a cada quadro.
- **Floresta (Extra Trees)**: centenas de árvores independentes com cortes sorteados; a resposta é a média dos votos.
- **Fonte (coleção)**: o conjunto de gravações de onde um exemplo veio. Aqui: Dryad (repositório público com gravações de várias espécies), HumBug (gravações de mosquitos feitas com celulares), ESC-50 (50 tipos de sons ambientais, usados como ruído), Funchal (Aedes e outros mosquitos gravados em vários aparelhos, Vasconcelos et al., 2020), MosquitoSong (gravações do estudo MosquitoSong+), UFRGS (Aedes gravados com aplicativos de celular, Paim et al., 2023) e Torres (espectrogramas de Aedes publicados por Torres et al., 2025).
- **Fonte da verdade**: o lugar que vale quando há divergência: os slides seguem este caderno, e não o contrário.
- **Frequência (Hz)**: quantas vezes por segundo algo se repete: 500 Hz = 500 ciclos por segundo.
- **Gradiente e otimizador (AdamW)**: o gradiente indica para onde mudar os pesos para errar menos; o otimizador é a regra que aplica a mudança.
- **Grupo**: conjunto de exemplos que não podem ficar dos dois lados da prova (ver §7).
- **Hiperparâmetro**: escolha feita antes do treino (número de árvores, taxa de aprendizado…), que não é ajustada pelos dados.
- **Histograma**: gráfico de barras que conta quantos exemplos caem em cada faixa de valores.
- **Importância por permutação**: embaralhar uma parte dos dados e medir quanto o resultado piora.
- **Impressão digital (hash, SHA-256)**: código curto calculado a partir do conteúdo: conteúdos iguais dão o mesmo código.
- **In situ**: no local real de uso (por exemplo, numa armadilha em campo).
- **Inferência**: usar o modelo já treinado para responder sobre uma gravação nova.
- **Interpretabilidade**: descobrir em que o modelo se baseia para decidir.
- **Janela**: trecho curto do áudio (cerca de 1 s) analisado de uma vez.
- **Janela de Hann**: suavização aplicada às bordas do trecho antes da FFT, para evitar artefatos.
- **Kernel RBF**: forma de medir a semelhança entre duas gravações que permite fronteiras curvas.
- **Limiar**: valor de corte da nota: acima dele, o modelo declara "Aedes".
- **LSTM e Bi-LSTM**: redes que leem uma sequência no tempo guardando memória do que veio antes; a Bi-LSTM lê nos dois sentidos.
- **Manifesto**: tabela que lista cada janela de áudio: arquivo, instante inicial, volume e fonte.
- **Margem e vetores de suporte**: margem é a distância entre a fronteira e os exemplos mais próximos; esses exemplos mais próximos são os vetores de suporte.
- **Matriz de confusão**: tabela que cruza o que era de verdade com o que o modelo disse.
- **Média geométrica (média dos logaritmos)**: média que só é alta se todos os valores forem altos.
- **Média por arquivo**: para arquivos com várias janelas, o exemplo é a média dos atributos de todas elas.
- **Memória da GPU (VRAM)**: memória da própria placa gráfica; o treino precisa que os dados e a rede caibam nela.
- **MLP**: rede de camadas de neurônios conectadas em sequência (perceptron multicamadas).
- **Modelo**: o programa que, depois de ajustado com exemplos, dá a resposta para gravações novas.
- **Mono**: áudio de um único canal (os estéreo são somados).
- **Neurônio, peso e parâmetro**: neurônio é uma soma ponderada seguida de uma função; os pesos (parâmetros) são os números ajustados no treino.
- **Normalização (z-score)**: subtrair a média e dividir pelo desvio padrão, deixando média 0 e desvio 1.
- **Nota (score)**: número de 0 a 1 que o modelo dá; quanto maior, mais ele acredita que é Aedes.
- **Otimista**: resultado que parece melhor do que será em dados novos.
- **p_m e p_a**: p_m = probabilidade de haver mosquito; p_a = probabilidade de ser Aedes, dado que é mosquito.
- **PCM**: a forma digital do áudio: a sequência de amostras numéricas.
- **Percentil e mediana**: o percentil 10 é o valor abaixo do qual estão 10% dos dados; a mediana é o percentil 50 (o valor do meio).
- **Peso de exemplo**: quanto cada exemplo conta no ajuste do modelo.
- **Precisão**: VP / (VP + FP): quando o modelo diz Aedes, quanto ele acerta.
- **Probabilidade (predict_proba)**: a função que devolve a probabilidade de cada classe.
- **Quadro**: cada instante (coluna) do espectrograma.
- **Reamostrar**: trocar a taxa de amostragem de um áudio.
- **Rede neural**: modelo feito de camadas de "neurônios", cada um somando entradas com pesos.
- **Regressão logística**: soma ponderada dos atributos passada por uma curva em S que a transforma em probabilidade.
- **ReLU**: função que zera valores negativos e mantém os positivos.
- **Repetição e média ± desvio**: repetimos a validação 3 vezes com sorteios diferentes; "0,86 ± 0,01" é a média e o quanto variou.
- **Reprodutível**: quem rodar de novo, com os mesmos dados e sementes, obtém o mesmo resultado.
- **Retrospectivo × teste cego**: retrospectivo = avaliado em dados que o projeto já conhecia; teste cego = avaliado em dados novos, escolhidos depois de congelar o método.
- **ROCm / HIP / CUDA**: plataformas que deixam o PyTorch usar a GPU: CUDA é a da NVIDIA; ROCm (e a interface HIP) é a da AMD.
- **Rótulo**: a resposta correta conhecida de um exemplo (aqui, a classe dele).
- **Semente (seed)**: número que fixa um sorteio aleatório: com a mesma semente, o sorteio sai igual e o resultado se repete.
- **STFT**: FFT repetida em janelinhas que deslizam pelo áudio; o resultado é o espectrograma.
- **SVM (máquina de vetores de suporte)**: procura a fronteira entre as classes com a maior folga (margem).
- **Tabela de atributos**: uma linha por gravação e 80 colunas de atributos, mais classe, fonte e grupo.
- **Taxa de amostragem (Hz, kHz)**: quantas medidas do som são guardadas por segundo. Em 8 kHz são 8.000 por segundo, e só cabem frequências até a metade disso, 4 kHz.
- **Tensor**: tabela numérica com várias dimensões (aqui: gravação × quadro × faixa).
- **Tom fundamental (f₀) e harmônicos**: f₀ é a frequência principal do zumbido (as batidas de asa por segundo); os harmônicos são múltiplos dela (2·f₀, 3·f₀…).
- **Treino e prova (teste)**: treino é o ajuste do modelo com exemplos; prova é a avaliação em exemplos que ele não viu.
- **União-busca (union-find)**: algoritmo que junta itens ligados entre si em conjuntos.
- **Validação cruzada**: dividir os dados em partes (dobras); treinar com algumas e provar nas outras, alternando até todas serem prova.
- **Vazamento**: quando informação da prova escapa para o treino e infla o resultado.
- **Volume (rms)**: raiz da média dos quadrados das amostras: mede o quão "alto" é um trecho.
- **VP, FP, FN, VN**: verdadeiro positivo (Aedes certo), falso positivo (alarme falso), falso negativo (Aedes perdido), verdadeiro negativo (não-Aedes certo).